# Sentiment-of-Sequence: Peer-Review Sync (2025-08-16)

**Bottom line.** Two oncogenic “grammars” emerged:
- **Context-dependent ($Δ$ wins):** EGFR, KRAS
- **Context-independent (WT neighborhood wins or must gate $Δ$):** TP53, PIK3CA

**Receipts (LOCO-GENE AUROC):**
- EGFR (n=39): `pair_twoR` **0.7257**, `rank-avg` 0.7029  
- KRAS (n=11): `pair_twoR` **0.7500** (high variance)  
- PIK3CA (n=24): `max(WT, pair)` **0.7778**, `WT-only` 0.7593, `pair` 0.3981  
- TP53 (n=197): `max(WT, pair)` **0.8020**, `rank-avg` 0.7191, `WT-only` 0.7160  

**Why it makes sense.**
- **Oncogenes (EGFR/KRAS):** specific changes flip function → $Δ$-CNN shines.
- **Tumor suppressor / mixed (TP53/PIK3CA):** regions are fragile by context → WT dominates; $Δ$ must be gated.

**Deploy rule (training-free, leak-free):**
- EGFR, KRAS → `pair_twoR`
- TP53, PIK3CA → `max(WT, pair)`
- Unknown gene fallback → `rank-avg`
- Add **per-gene calibration**: isotonic for WT-heavy, beta for Δ-heavy.

**Diagnostics we trust (auto-gates):**
- **Tri-coverage** and **tri-frequency shift** (PIK3CA’s dip = frequency skew, not OOD).
- **Grammar stability score**: AUROC vs #negatives; down-weight Δ when unstable.
- Always report **bootstrap CIs**.

**Next moves (priority):**
1. Add scalar features (WT-safe): codon position, local GC%, palindromic/repeats, splice distance, strand bit.
2. Upgrade negatives: add **COSMIC passengers** → run 3-class (driver/passenger/germline).
3. Strand/asymmetry check: 5’ vs 3’ ablations; verify $Δ$ dependence for EGFR/KRAS.
4. Saliency maps: WT-k-mer importances (TP53) vs $Δ$ saliency (EGFR); connect to COSMIC SBS signatures.
5. Light hard-negative mining (train folds only, K≤2); re-eval two-radius Δ ensemble.

**Artifacts (from 2025-08-16):**
- Pair ensemble: `artifacts/ensemble_twoR_20250816_225822/metrics_twoR_mix.csv`
- WT baseline: `artifacts/wt_only_baseline_20250816_230700/metrics_wt_only.csv`
- Blends: `artifacts/blend_nolearn_20250816_231849/metrics_blends.csv`
- Unified receipts: `artifacts/final_receipts_20250816_232227/`

**Open questions:**
- When to **always** gate $Δ$ with WT vs using tri-frequency flags?
- Does radius still matter after calibration/blending?
- Minimal negatives per gene for Δ stability (early sign: ~30–40)?


In [ ]:
# ---- Colab env check + A100-optimized knobs ----
import os, platform, subprocess, re, torch

# 1) Show the GPU Colab actually gave us
try:
    out = subprocess.check_output(
        ["bash","-lc","nvidia-smi --query-gpu=name,memory.total,driver_version,cuda_version --format=csv,noheader"],
        stderr=subprocess.STDOUT, text=True
    ).strip()
    print("nvidia-smi:", out)
except Exception as e:
    print("nvidia-smi: None (CPU or TPU?)", e)

# 2) Pick device
if "COLAB_TPU_ADDR" in os.environ:
    DEVICE = "tpu"     # we'll avoid TPU today unless we switch to TF/JAX
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

name = torch.cuda.get_device_name(0) if DEVICE=="cuda" else platform.processor()
vram = (torch.cuda.get_device_properties(0).total_memory/(1024**3)) if DEVICE=="cuda" else None
print(f"Device: {DEVICE} | Name: {name} | VRAM: {vram:.1f} GB" if vram else f"Device: {DEVICE} | Name: {name}")

# 3) A100-friendly perf toggles (safe for L4/T4 too)
if DEVICE=="cuda":
    torch.set_float32_matmul_precision("high")  # enable TF32 kernels
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True       # if input shapes are stable
    AMP_DTYPE = torch.bfloat16                  # A100/L4 support BF16 natively
else:
    AMP_DTYPE = None

# 4) Helpers you’ll reuse below
def compile_model(m):
    try:
        return torch.compile(m, mode="max-autotune")
    except Exception:
        return m  # fallback if torch.compile not happy

def mk_loader(ds, batch_size):
    from torch.utils.data import DataLoader
    import os
    return DataLoader(
        ds, batch_size=batch_size, shuffle=True,
        num_workers=max(2, (os.cpu_count() or 4)//2),
        pin_memory=(DEVICE=="cuda"), persistent_workers=True, prefetch_factor=4
    )

# Heuristic starting points for this project (bump until OOM, then back off ~15%)
BATCH_WT   = 4096 if DEVICE=="cuda" else 256      # WT baseline
BATCH_DCNN = 1024 if DEVICE=="cuda" else 64       # Δ-CNN windows
print(f"AMP dtype: {AMP_DTYPE} | BATCH_WT={BATCH_WT} | BATCH_DCNN={BATCH_DCNN}")


today’s targets:

1. **Re-open yesterday’s receipts** → detect each gene’s **grammar type** (context-dependent vs context-independent) straight from metrics.

2. **Set the per-gene deploy rule** (WT vs $Δ$ or max(WT,$Δ$)) and show a clean summary table.

3. (then) **Calibrate per gene** (isotonic for WT-heavy, beta for $Δ$-heavy) and re-eval.

### Block 1 — Mount + scan artifacts → auto-detect grammar type

We will now:

* Mount Drive

* Hunt for your receipts (glob under common paths, including your `/content/drive/.../artifacts/final_receipts_20250816_232227`)

* Parse metrics, pick best WT vs best $Δ$ (pair) per gene

* Label grammar type and suggest the strategy

In [ ]:
# --- Block 1: receipts scan + grammar-type detection ---
import os, re, glob, json, math
import pandas as pd
from google.colab import drive

# 0) Mount Drive (safe if already mounted)
try:
    drive.mount('/content/drive', force_remount=False)
except Exception as e:
    print("Drive mount note:", e)

# 1) Candidate roots to search for artifacts/metrics
CANDIDATE_ROOTS = [
    "/content/drive/MyDrive",                                    # your Drive
    "/content/drive/MyDrive/Cancer_gene_database",               # project dir seen yesterday
    "/content",                                                  # local session
]
# You can add a precise path here if you know it:
HINT = "/content/drive/MyDrive/Cancer_gene_database/artifacts"   # tweak if yours differs

roots = [p for p in [HINT] + CANDIDATE_ROOTS if os.path.exists(p)]

# 2) Find metrics CSVs
csvs = []
for r in roots:
    csvs += glob.glob(os.path.join(r, "**", "metrics*.csv"), recursive=True)
csvs = sorted(set(csvs))
print(f"Found {len(csvs)} metrics files")
for p in csvs[:8]:
    print(" •", p)

if not csvs:
    raise SystemExit("No metrics*.csv found. Update HINT to your artifacts dir and re-run.")

# 3) Load and normalize
def load_metrics(path):
    try:
        df = pd.read_csv(path)
        df["__src"] = path
        return df
    except Exception as e:
        print(f"Skip {path}: {e}")
        return None

frames = [load_metrics(p) for p in csvs]
frames = [f for f in frames if f is not None and not f.empty]
metrics = pd.concat(frames, ignore_index=True)
# Normalize column names
metrics.columns = [c.strip().lower() for c in metrics.columns]

# Heuristic harmonization
rename_map = {
    "pr_auc": "prauc",
    "pr-auc": "prauc",
    "au_pr": "prauc",
}
metrics.rename(columns=rename_map, inplace=True)
# Fill expected columns
for col in ["gene","method","auroc","prauc","brier","n"]:
    if col not in metrics.columns:
        metrics[col] = None

# 4) Tag "WT" vs "PAIR/Δ" vs "BLEND"
def method_kind(m):
    m = (m or "").lower()
    if any(k in m for k in ["pair", "delta", "Δ", "siamese", "twor", "dcnn", "ranknet"]):
        return "pair"
    if any(k in m for k in ["wt-only","wt_only","wt baseline","wtonly","wtonlybaseline","wt"]):
        # ensure we don't misclassify blends like "max(WT,pair)"
        if "max" in m or "blend" in m or "rank-avg" in m or "avg" in m:
            return "blend"
        return "wt"
    if any(k in m for k in ["blend","rank-avg","logit","max(wt","max_wt","max(", "ensemble"]):
        return "blend"
    return "other"

metrics["kind"] = metrics["method"].apply(method_kind)

# Optional: filter to LOCO-GENE only if a split column exists
if "split" in metrics.columns:
    mask = metrics["split"].str.contains("loco", case=False, na=False)
    metrics = metrics[mask].copy()

# 5) Best WT vs best Pair per gene
def best_by_kind(df, kind):
    sub = df[df["kind"]==kind]
    if sub.empty:
        return None
    sub = sub.sort_values(by=["auroc"], ascending=False)
    return sub.iloc[0][["gene","method","auroc","prauc","brier","__src"]]

genes = sorted([g for g in metrics["gene"].dropna().unique().tolist() if g])

rows = []
for g in genes:
    gdf = metrics[metrics["gene"]==g]
    wt  = best_by_kind(gdf, "wt")
    pr  = best_by_kind(gdf, "pair")
    bl  = best_by_kind(gdf, "blend")

    wt_auc  = float(wt["auroc"]) if wt is not None and pd.notna(wt["auroc"]) else float("nan")
    pr_auc  = float(pr["auroc"]) if pr is not None and pd.notna(pr["auroc"]) else float("nan")

    # Grammar-type heuristic:
    # - context_independent if WT >= 0.65 and Pair is not higher by >= 0.03
    # - else context_dependent
    if not math.isnan(wt_auc) and wt_auc >= 0.65 and (math.isnan(pr_auc) or pr_auc < wt_auc + 0.03):
        grammar = "context_independent"
        strategy = "max(WT, pair)"  # gate Δ with WT
        calib = "isotonic_on_WT"
        weight_note = "0.8*WT + 0.2*Δ (start)"
    else:
        grammar = "context_dependent"
        strategy = "pair_twoR_or_best_pair"
        calib = "beta_on_pair"
        weight_note = "0.2*WT + 0.8*Δ (start)"

    rows.append({
        "gene": g,
        "best_wt_auc": round(wt_auc,4) if not math.isnan(wt_auc) else None,
        "best_wt_method": (None if wt is None else wt["method"]),
        "best_pair_auc": round(pr_auc,4) if not math.isnan(pr_auc) else None,
        "best_pair_method": (None if pr is None else pr["method"]),
        "grammar_type": grammar,
        "recommended_strategy": strategy,
        "calibration": calib,
        "blend_hint": weight_note,
        "metrics_src": (None if wt is None else wt["__src"])
    })

summary = pd.DataFrame(rows).sort_values(by=["gene"]).reset_index(drop=True)
print("\n=== Grammar-type summary ===")
display(summary)

# 6) Quick assert: flag suspicious cases (no WT or no Pair seen)
issues = summary[(summary.best_wt_auc.isna()) | (summary.best_pair_auc.isna())]
if not issues.empty:
    print("\n[Heads-up] Some genes missing WT or Pair metrics. Will fall back to available:")
    display(issues[["gene","best_wt_auc","best_pair_auc"]])

# Save snapshot to Drive for lineage
outdir = "/content/grammar_scan"
os.makedirs(outdir, exist_ok=True)
summary_path = os.path.join(outdir, "grammar_type_summary.csv")
summary.to_csv(summary_path, index=False)
print("\nSaved:", summary_path)


In [ ]:
# --- Block 1 (robust): receipts scan + grammar-type detection ---
import os, re, glob, json, math
import pandas as pd
from google.colab import drive

# 0) Mount Drive (no-op if already mounted)
try:
    drive.mount('/content/drive', force_remount=False)
except Exception as e:
    print("Drive mount note:", e)

# 1) Where to look
CANDIDATE_ROOTS = [
    "/content/drive/MyDrive/Cancer_gene_database/artifacts",
    "/content/drive/MyDrive",
    "/content",
]

# 2) Find metrics CSVs
csvs = []
for r in CANDIDATE_ROOTS:
    if os.path.exists(r):
        csvs += glob.glob(os.path.join(r, "**", "metrics*.csv"), recursive=True)
csvs = sorted(set(csvs))
print(f"Found {len(csvs)} metrics files")
for p in csvs[:8]:
    print(" •", p)
if not csvs:
    raise SystemExit("No metrics*.csv found. Point CANDIDATE_ROOTS to your artifacts dir and re-run.")

# 3) Load + normalize schemas
def load_metrics(path):
    try:
        df = pd.read_csv(path)
        df["__src"] = path
        return df
    except Exception as e:
        print(f"[skip] {path}: {e}")
        return None

frames = [f for f in (load_metrics(p) for p in csvs) if f is not None and not f.empty]
metrics = pd.concat(frames, ignore_index=True)
metrics.columns = [c.strip().lower() for c in metrics.columns]

# 3a) Column harmonization
rename_map = {"pr_auc":"prauc", "pr-auc":"prauc", "au_pr":"prauc"}
metrics.rename(columns=rename_map, inplace=True)

# Ensure key columns exist (fill from alternates or defaults)
def first_col(dframe, options, default_name, default_value=None):
    for c in options:
        if c in dframe.columns:
            return dframe[c]
    return pd.Series([default_value]*len(dframe), index=dframe.index, name=default_name)

metrics["method"] = first_col(metrics,
    ["method","model","approach","algorithm","name","mode","variant"], "method", ""
).astype(str).fillna("")

metrics["gene"] = first_col(metrics,
    ["gene","target_gene","locus_gene"], "gene", ""
).astype(str).fillna("")

for col in ["auroc","prauc","brier","n","n_test","count"]:
    if col in metrics.columns:
        metrics[col] = pd.to_numeric(metrics[col], errors="coerce")

# 4) Tag "WT" vs "PAIR/Δ" vs "BLEND"
def method_kind(m):
    m = str(m or "").lower()
    if any(k in m for k in ["pair","delta","Δ","siamese","twor","dcnn","ranknet"]):
        return "pair"
    if any(k in m for k in ["max(wt","blend","rank-avg","logit","ensemble","avg"]):
        return "blend"
    if "wt" in m and "pair" not in m and "blend" not in m:
        return "wt"
    return "other"

metrics["kind"] = metrics["method"].apply(method_kind)

# 5) Prefer LOCO rows if present
mask_candidates = []
if "split" in metrics.columns:
    mask_candidates.append(metrics["split"].str.contains("loco", case=False, na=False))
if "tag" in metrics.columns:
    mask_candidates.append(metrics["tag"].str.contains("loco", case=False, na=False))
use_mask = None
if mask_candidates:
    use_mask = mask_candidates[0]
    for m in mask_candidates[1:]:
        use_mask = use_mask | m
if use_mask is not None and use_mask.any():
    metrics = metrics[use_mask].copy()

# 6) Best WT vs best Pair per gene
def best_by_kind(df, kind):
    sub = df[df["kind"]==kind]
    if sub.empty:
        return None
    sub = sub.sort_values(by=["auroc"], ascending=False)
    cols = [c for c in ["gene","method","auroc","prauc","brier","__src"] if c in sub.columns]
    return sub.iloc[0][cols]

genes = sorted([g for g in metrics["gene"].dropna().unique().tolist() if g])
rows = []
for g in genes:
    gdf = metrics[metrics["gene"]==g]
    wt = best_by_kind(gdf, "wt")
    pr = best_by_kind(gdf, "pair")
    wt_auc = float(wt["auroc"]) if wt is not None and pd.notna(wt["auroc"]) else float("nan")
    pr_auc = float(pr["auroc"]) if pr is not None and pd.notna(pr["auroc"]) else float("nan")

    # Heuristic: WT >= 0.65 and Pair not ≥ WT+0.03 → context_independent; else context_dependent
    if not math.isnan(wt_auc) and wt_auc >= 0.65 and (math.isnan(pr_auc) or pr_auc < wt_auc + 0.03):
        grammar = "context_independent"
        strategy = "max(WT, pair)"       # gate Δ with WT
        calib = "isotonic_on_WT"
        weight_note = "0.8*WT + 0.2*Δ (start)"
    else:
        grammar = "context_dependent"
        strategy = "pair_twoR_or_best_pair"
        calib = "beta_on_pair"
        weight_note = "0.2*WT + 0.8*Δ (start)"

    rows.append({
        "gene": g,
        "best_wt_auc": None if math.isnan(wt_auc) else round(wt_auc,4),
        "best_wt_method": None if wt is None else wt.get("method"),
        "best_pair_auc": None if math.isnan(pr_auc) else round(pr_auc,4),
        "best_pair_method": None if pr is None else pr.get("method"),
        "grammar_type": grammar,
        "recommended_strategy": strategy,
        "calibration": calib,
        "blend_hint": weight_note,
        "metrics_src": None if wt is None else wt.get("__src"),
    })

summary = pd.DataFrame(rows).sort_values("gene").reset_index(drop=True)
print("\n=== Grammar-type summary ===")
display(summary)

# 7) Flag missing kinds
issues = summary[(summary.best_wt_auc.isna()) | (summary.best_pair_auc.isna())]
if not issues.empty:
    print("\n[Heads-up] Some genes missing WT or Pair metrics. Falling back to what's available:")
    display(issues[["gene","best_wt_auc","best_pair_auc"]])

# 8) Save
outdir = "/content/grammar_scan"
os.makedirs(outdir, exist_ok=True)
summary_path = os.path.join(outdir, "grammar_type_summary.csv")
summary.to_csv(summary_path, index=False)
print("\nSaved:", summary_path)


In [ ]:
# --- Block 1 (schema-sniffer): receipts scan → grammar-type detection (robust) ---
import os, re, glob, math
import pandas as pd
from google.colab import drive

# 0) Mount (no-op if already mounted)
try:
    drive.mount('/content/drive', force_remount=False)
except Exception as e:
    print("Drive mount note:", e)

# 1) Search roots
ROOTS = [
    "/content/drive/MyDrive/Cancer_gene_database/artifacts",
    "/content/drive/MyDrive",
    "/content",
]
csvs = []
for r in ROOTS:
    if os.path.exists(r):
        csvs += glob.glob(os.path.join(r, "**", "metrics*.csv"), recursive=True)
csvs = sorted(set(csvs))
print(f"Found {len(csvs)} metrics files")
for p in csvs[:8]: print(" •", p)
if not csvs:
    raise SystemExit("No metrics*.csv found.")

def load_csv(p):
    try:
        df = pd.read_csv(p)
        df["__src"] = p
        return df
    except Exception as e:
        print(f"[skip] {p}: {e}")
        return None

frames = [x for x in (load_csv(p) for p in csvs) if x is not None and not x.empty]
if not frames:
    raise SystemExit("No readable metrics CSVs.")
raw = pd.concat(frames, ignore_index=True)
raw.columns = [c.strip().lower() for c in raw.columns]

# 2) Quick schema peek (first 3 files) — helps debugging if needed
print("\n[Schema peek]")
for p in csvs[:3]:
    df = load_csv(p);
    if df is not None:
        print(" •", p)
        print("   cols:", [c for c in df.columns])
        print(df.head(2).to_string(index=False))

# 3) Harmonize columns (best-effort)
df = raw.copy()

# AUC columns (try multiple spellings)
def coalesce_num(row, candidates):
    for c in candidates:
        if c in row and pd.notna(row[c]):
            try:
                return float(row[c])
            except Exception:
                pass
    return float("nan")

auc_candidates   = ["auroc","roc_auc","auc_roc","au_roc","roc-auc","rocauc","roc_auc_mean","auc"]
prauc_candidates = ["prauc","pr_auc","au_pr","pr-auc","average_precision","ap"]
brier_candidates = ["brier","brier_score","brier_loss"]

# Method/gene discovery
method_candidates = ["method","model","approach","algorithm","name","mode","variant","estimator"]
gene_candidates   = ["gene","gene_symbol","target_gene","locus_gene","locus","target","symbol"]

def extract_method(row):
    for c in method_candidates:
        if c in row and isinstance(row[c], str) and row[c].strip():
            return row[c]
    # fallback: infer from path
    src = row.get("__src","").lower()
    if "blend" in src: return "blend_from_path"
    if "twor" in src or "delta" in src or "pair" in src: return "pair_from_path"
    if "wt" in src or "base" in src: return "wt_from_path"
    return ""

GENE_WHITELIST = {"EGFR","KRAS","PIK3CA","TP53"}  # expand as you add genes

gene_tag_patterns = [
    re.compile(r"(?:loco[_-]?gene:|gene:)\s*([A-Za-z0-9]+)", re.I),
    re.compile(r"(?:loco[_-]?site:)\s*([A-Za-z0-9]+)", re.I),
]

def extract_gene(row):
    # direct columns first
    for c in gene_candidates:
        if c in row and isinstance(row[c], str) and row[c].strip():
            return row[c].strip().upper()
    # look into 'tag' or similar text cols
    for c in ["tag","fold","split","notes","comment","group"]:
        if c in row and isinstance(row[c], str) and row[c]:
            txt = row[c]
            # pattern like "loco_gene:EGFR"
            for pat in gene_tag_patterns:
                m = pat.search(txt)
                if m:
                    return m.group(1).upper()
            # fallback: whitelist scan
            toks = re.findall(r"[A-Za-z0-9]+", txt.upper())
            for t in toks:
                if t in GENE_WHITELIST:
                    return t
    # last resort: path whitelist
    src = row.get("__src","").upper()
    for g in GENE_WHITELIST:
        if g in src:
            return g
    return ""

# Build normalized rows
norm_rows = []
for idx, r in df.iterrows():
    rdict = r.to_dict()
    auroc = coalesce_num(rdict, auc_candidates)
    prauc = coalesce_num(rdict, prauc_candidates)
    brier = coalesce_num(rdict, brier_candidates)
    method = extract_method(rdict)
    gene = extract_gene(rdict)
    norm_rows.append({
        "gene": gene or None,
        "method": method or None,
        "auroc": None if math.isnan(auroc) else auroc,
        "prauc": None if math.isnan(prauc) else prauc,
        "brier": None if math.isnan(brier) else brier,
        "__src": rdict.get("__src","")
    })

metrics = pd.DataFrame(norm_rows).dropna(subset=["gene"], how="any")
metrics = metrics[metrics["gene"].isin(GENE_WHITELIST)].copy()

# If we still have nothing, abort with a helpful hint
if metrics.empty:
    print("\n[No normalized metrics for EGFR/KRAS/PIK3CA/TP53]")
    print("Columns present across files:", sorted(set(raw.columns)))
    raise SystemExit("Update GENE_WHITELIST or column maps above and re-run.")

# 4) Classify kind (WT / Pair / Blend / Other)
def method_kind(m, src):
    s = (m or "").lower() + " " + (src or "").lower()
    if any(k in s for k in ["blend","rank-avg","logit","ensemble","max(wt"]):
        return "blend"
    if any(k in s for k in ["pair","delta","Δ","siamese","twor","dcnn","ranknet"]):
        return "pair"
    if "wt" in s and "pair" not in s and "blend" not in s:
        return "wt"
    if any(k in s for k in ["base","baseline"]) and "pair" not in s:
        return "wt"
    return "other"

metrics["kind"] = [method_kind(m, s) for m, s in zip(metrics["method"], metrics["__src"])]

# Prefer LOCO rows if a split/tag exists (already baked into extraction),
# so we just proceed to best-per-kind per gene.

def best_by_kind(gdf, kind):
    sub = gdf[gdf["kind"]==kind]
    if sub.empty: return None
    return sub.sort_values("auroc", ascending=False).iloc[0]

rows = []
for g, gdf in metrics.groupby("gene"):
    wt  = best_by_kind(gdf, "wt")
    pr  = best_by_kind(gdf, "pair")
    wt_auc = None if wt is None else wt["auroc"]
    pr_auc = None if pr is None else pr["auroc"]

    # Heuristic: WT >= 0.65 and Pair not ≥ WT+0.03 → context_independent
    if wt_auc is not None and wt_auc >= 0.65 and (pr_auc is None or pr_auc < wt_auc + 0.03):
        grammar = "context_independent"
        strategy = "max(WT, pair)"
        calib = "isotonic_on_WT"
        blend = "0.8*WT + 0.2*Δ (start)"
    else:
        grammar = "context_dependent"
        strategy = "pair_twoR_or_best_pair"
        calib = "beta_on_pair"
        blend = "0.2*WT + 0.8*Δ (start)"

    rows.append({
        "gene": g,
        "best_wt_auc": None if wt is None else round(float(wt_auc),4),
        "best_wt_method": None if wt is None else wt["method"],
        "best_pair_auc": None if pr is None else round(float(pr_auc),4),
        "best_pair_method": None if pr is None else pr["method"],
        "grammar_type": grammar,
        "recommended_strategy": strategy,
        "calibration": calib,
        "blend_hint": blend,
        "wt_src": None if wt is None else wt["__src"],
        "pair_src": None if pr is None else pr["__src"],
    })

summary = pd.DataFrame(rows).sort_values("gene").reset_index(drop=True)
print("\n=== Grammar-type summary ===")
display(summary)

# Flag holes
holes = summary[(summary.best_wt_auc.isna()) | (summary.best_pair_auc.isna())]
if not holes.empty:
    print("\n[Heads-up] Missing WT or Pair in some genes (will fallback to available):")
    display(holes[["gene","best_wt_auc","best_pair_auc","wt_src","pair_src"]])

# Save snapshot
outdir = "/content/grammar_scan"
os.makedirs(outdir, exist_ok=True)
summary_path = os.path.join(outdir, "grammar_type_summary.csv")
summary.to_csv(summary_path, index=False)
print("\nSaved:", summary_path)


### Block 1.1 — schema-aware normalization (wide→long) + clean grammar detect

our above pass grabbed the wrong columns in a couple files (mixed AUROC/PR spellings + wide “AUROC_pair / AUROC_wt” tables). Fix it properly: pivot the wide files to long rows per method, and never coalesce PR into ROC.

In [ ]:
# --- Block 1.1: normalize metrics (robust) + grammar-type detection ---
import os, re, glob, math
import pandas as pd
from google.colab import drive
from IPython.display import display

# 0) Mount (no-op if already mounted)
try:
    drive.mount('/content/drive', force_remount=False)
except Exception as e:
    print("Drive mount note:", e)

# 1) Find all metrics*.csv
ROOTS = [
    "/content/drive/MyDrive/Cancer_gene_database/artifacts",
    "/content/drive/MyDrive",
    "/content",
]
csvs = []
for r in ROOTS:
    if os.path.exists(r):
        csvs += glob.glob(os.path.join(r, "**", "metrics*.csv"), recursive=True)
csvs = sorted(set(csvs))
print(f"Found {len(csvs)} metrics files")
for p in csvs[:8]: print(" •", p)
if not csvs:
    raise SystemExit("No metrics*.csv found.")

# --- helpers ---
GENES = {"EGFR","KRAS","PIK3CA","TP53"}

def infer_method_from_path(src: str) -> str:
    s = (src or "").lower()
    if "ensemble" in s or "twor" in s:
        return "pair_twoR"
    if "blend" in s:
        return "blend"
    if any(k in s for k in ["delta","pair"]):
        return "pair"
    if any(k in s for k in ["wt","base"]):
        return "wt"
    return ""

def normalize_df(df: pd.DataFrame, src: str) -> pd.DataFrame:
    """Return rows with columns: fold,gene,n,method,auroc,prauc,brier,__src"""
    if df is None or df.empty:
        return pd.DataFrame()
    d = df.copy()
    d.columns = [c.strip().lower() for c in d.columns]
    d["__src"] = src

    base_cols = [c for c in ["fold","gene","n"] if c in d.columns]

    # Case A: wide metrics like auroc_pair, auroc_wt, auroc_rankavg, pr_pair, brier_pair...
    auroc_wide = [c for c in d.columns if c.startswith("auroc_")]
    if auroc_wide:
        # group columns by suffix after first underscore
        groups = {}
        for c in d.columns:
            if c.startswith("auroc_"):
                sfx = c.split("_", 1)[1]
                groups.setdefault(sfx, {})["auroc"] = c
            elif c.startswith("pr_"):
                sfx = c.split("_", 1)[1]
                groups.setdefault(sfx, {})["prauc"] = c
            elif c.startswith("prauc_"):
                sfx = c.split("_", 1)[1]
                groups.setdefault(sfx, {})["prauc"] = c
            elif c.startswith("brier_"):
                sfx = c.split("_", 1)[1]
                groups.setdefault(sfx, {})["brier"] = c

        out_frames = []
        for sfx, cols in groups.items():
            out = d[base_cols].copy() if base_cols else pd.DataFrame(index=d.index)
            out["method"] = sfx  # 'pair', 'wt', 'rankavg', 'logitavg', 'max', ...
            out["auroc"] = pd.to_numeric(d[cols.get("auroc")], errors="coerce") if "auroc" in cols else None
            out["prauc"] = pd.to_numeric(d[cols.get("prauc")], errors="coerce") if "prauc" in cols else None
            out["brier"] = pd.to_numeric(d[cols.get("brier")], errors="coerce") if "brier" in cols else None
            out["__src"] = src
            out_frames.append(out)
        return pd.concat(out_frames, ignore_index=True)

    # Case B: long metrics with 'auroc' present (plus maybe 'method')
    if "auroc" in d.columns:
        out = d.copy()
        if "method" not in out.columns or out["method"].isna().all():
            out["method"] = infer_method_from_path(src)
        if "pr_auc" in out.columns and "prauc" not in out.columns:
            out.rename(columns={"pr_auc":"prauc"}, inplace=True)
        for col in ["prauc","brier"]:
            if col not in out.columns:
                out[col] = None
        keep = [c for c in ["fold","gene","n","method","auroc","prauc","brier","__src"] if c in out.columns]
        return out[keep]

    # else nothing we can use
    return pd.DataFrame()

def load_csv(p):
    try:
        return pd.read_csv(p)
    except Exception as e:
        print(f"[skip] {p}: {e}")
        return None

# 2) Normalize all files
norm_frames = []
for p in csvs:
    df = load_csv(p)
    nf = normalize_df(df, p)
    if not nf.empty:
        norm_frames.append(nf)

norm = pd.concat(norm_frames, ignore_index=True) if norm_frames else pd.DataFrame()
if norm.empty:
    raise SystemExit("Could not normalize any metrics.")

# 3) Make sure GENE is filled (from fold or path if missing)
def fill_gene(row):
    g = str(row.get("gene") or "").strip().upper()
    if g:
        return g
    fold = str(row.get("fold") or "")
    m = re.search(r"gene:([A-Za-z0-9]+)", fold, flags=re.I)
    if m:
        return m.group(1).upper()
    src = str(row.get("__src") or "").upper()
    for gg in GENES:
        if gg in src:
            return gg
    return None

norm["gene"] = norm.apply(fill_gene, axis=1)
norm = norm[norm["gene"].isin(GENES)].copy()

# 4) Classify method kind
def kind_of(method: str) -> str:
    m = (method or "").lower()
    if m in {"pair","pair_twoR","twor","delta","ranknet","siamese"} or "pair" in m or "delta" in m:
        return "pair"
    if m in {"wt","wtonly","wt_only","baseline"} or ("wt" in m and "pair" not in m and "blend" not in m):
        return "wt"
    if m in {"rankavg","logitavg","max","blend"} or "blend" in m or "avg" in m or "max" in m:
        return "blend"
    return "other"

norm["kind"] = norm["method"].apply(kind_of)

# 5) Best WT vs best Pair per gene (by AUROC)
def best_by_kind(gdf, kind):
    sub = gdf[(gdf["kind"]==kind) & pd.notna(gdf["auroc"])]
    if sub.empty:
        return None
    return sub.sort_values("auroc", ascending=False).iloc[0]

rows = []
for g, gdf in norm.groupby("gene"):
    wt = best_by_kind(gdf, "wt")
    pr = best_by_kind(gdf, "pair")
    wt_auc = None if wt is None else float(wt["auroc"])
    pr_auc = None if pr is None else float(pr["auroc"])

    # Heuristic: WT >= 0.65 and Pair not ≥ WT+0.03 → context_independent
    if wt_auc is not None and wt_auc >= 0.65 and (pr_auc is None or pr_auc < wt_auc + 0.03):
        grammar = "context_independent"
        strategy = "max(WT, pair)"
        calib = "isotonic_on_WT"
        blend = "0.8*WT + 0.2*Δ (start)"
    else:
        grammar = "context_dependent"
        strategy = "pair_twoR_or_best_pair"
        calib = "beta_on_pair"
        blend = "0.2*WT + 0.8*Δ (start)"

    rows.append({
        "gene": g,
        "best_wt_auc": None if wt is None else round(wt_auc,4),
        "best_wt_method": None if wt is None else wt["method"],
        "best_pair_auc": None if pr is None else round(pr_auc,4),
        "best_pair_method": None if pr is None else pr["method"],
        "grammar_type": grammar,
        "recommended_strategy": strategy,
        "calibration": calib,
        "blend_hint": blend,
        "wt_src": None if wt is None else wt["__src"],
        "pair_src": None if pr is None else pr["__src"],
    })

summary = pd.DataFrame(rows).sort_values("gene").reset_index(drop=True)

print("\n=== Grammar-type summary (clean) ===")
display(summary)

# 6) Persist for the next block
os.makedirs("/content/grammar_scan", exist_ok=True)
summary_path = "/content/grammar_scan/grammar_type_summary.csv"
norm_path = "/content/grammar_scan/metrics_long.csv"
summary.to_csv(summary_path, index=False)
norm.to_csv(norm_path, index=False)
print("Saved:", summary_path, "and", norm_path)


we got a table, but i’m side-eyeing those WT=1.0000 vibes on KRAS. smells like either small-n or we mixed splits. let’s lock this to LOCO-GENE only so we’re comparing apples to apples, then we’ll set the deploy rule.

### Block 1.2 — constrain to LOCO-GENE and recompute summary

In [ ]:
# --- Block 1.2: LOCO-GENE filter → clean grammar summary ---
import re, math, pandas as pd
from IPython.display import display

LONG_PATH = "/content/grammar_scan/metrics_long.csv"
norm = pd.read_csv(LONG_PATH)

# ensure columns exist
for c in ["fold","method","gene","auroc","__src"]:
    if c not in norm.columns: norm[c] = ""

# keep only our genes
GENES = {"EGFR","KRAS","PIK3CA","TP53"}
norm["gene"] = norm["gene"].astype(str).str.upper()
norm = norm[norm["gene"].isin(GENES)].copy()

# LOCO mask: prefer loco_gene; fallback to any 'loco' in fold or path
loco_mask = norm["fold"].str.contains(r"\bloco[_-]?gene\b", case=False, na=False)
if not loco_mask.any():
    loco_mask = norm["fold"].str.contains("loco", case=False, na=False) | norm["__src"].str.contains("loco", case=False, na=False)
loco = norm[loco_mask].copy()
if loco.empty:
    print("[warn] No LOCO rows found — using all rows (might inflate WT).")
    loco = norm.copy()

# method kind
def kind_of(m: str) -> str:
    m = (m or "").lower()
    if any(k in m for k in ["pair","delta","twor","ranknet","siamese"]):
        return "pair"
    if ("wt" in m and "pair" not in m and "blend" not in m) or m in {"baseline","wt_only","wtonly"}:
        return "wt"
    if any(k in m for k in ["blend","rankavg","logitavg","max","avg"]):
        return "blend"
    return "other"

loco["kind"] = loco["method"].astype(str).apply(kind_of)
loco["auroc"] = pd.to_numeric(loco["auroc"], errors="coerce")

def best_by_kind(gdf, kind):
    sub = gdf[(gdf["kind"]==kind) & pd.notna(gdf["auroc"])]
    return None if sub.empty else sub.sort_values("auroc", ascending=False).iloc[0]

rows=[]
for g, gdf in loco.groupby("gene"):
    wt = best_by_kind(gdf, "wt")
    pr = best_by_kind(gdf, "pair")
    wt_auc = None if wt is None else float(wt["auroc"])
    pr_auc = None if pr is None else float(pr["auroc"])
    # heuristic: WT ≥ 0.65 and Pair not ≥ WT+0.03 → context_independent
    if wt_auc is not None and wt_auc >= 0.65 and (pr_auc is None or pr_auc < wt_auc + 0.03):
        grammar, strategy, calib, blend = "context_independent", "max(WT, pair)", "isotonic_on_WT", "0.8*WT + 0.2*Δ (start)"
    else:
        grammar, strategy, calib, blend = "context_dependent", "pair_twoR_or_best_pair", "beta_on_pair", "0.2*WT + 0.8*Δ (start)"
    rows.append({
        "gene": g,
        "best_wt_auc": None if wt is None else round(wt_auc,4),
        "best_wt_method": None if wt is None else wt["method"],
        "best_pair_auc": None if pr is None else round(pr_auc,4),
        "best_pair_method": None if pr is None else pr["method"],
        "grammar_type": grammar,
        "recommended_strategy": strategy,
        "calibration": calib,
        "blend_hint": blend,
        "wt_src": None if wt is None else wt["__src"],
        "pair_src": None if pr is None else pr["__src"],
    })

summary_loco = pd.DataFrame(rows).sort_values("gene").reset_index(drop=True)
display(summary_loco)

# save
summary_path = "/content/grammar_scan/grammar_type_summary_loco.csv"
summary_loco.to_csv(summary_path, index=False)
print("Saved:", summary_path)


* **EGFR**: WT 0.6057 vs **PAIR 0.7800** → context-dependent → run **pair_twoR** (+ beta-cal later)

**KRAS:** WT 0.4167 vs **PAIR 0.8750** → **context-dependent** → run **pair_twoR** (+ beta-cal later)

**PIK3CA**: WT 0.7593 vs **pair 0.7222** → **context-independent** → **max(WT, $Δ$)** (+ isotonic on WT later)

**TP53**: WT 0.7160 vs **pair 0.6470** → **context-independent** → **max(WT, $Δ$)** (+ isotonic on WT later)

### --- Block 2: per-gene deploy config + simple combiner ---


In [ ]:
# --- Block 2: per-gene deploy config + simple combiner ---
import json, os
import pandas as pd

SUMMARY = "/content/grammar_scan/grammar_type_summary_loco.csv"
df = pd.read_csv(SUMMARY)

# lock the plan based on your table
deploy_cfg = {
    "EGFR":  {"grammar":"context_dependent",   "strategy":"pair", "calibration":"beta_on_pair",      "blend_weights":{"wt":0.2,"pair":0.8}},
    "KRAS":  {"grammar":"context_dependent",   "strategy":"pair", "calibration":"beta_on_pair",      "blend_weights":{"wt":0.2,"pair":0.8}},
    "PIK3CA":{"grammar":"context_independent", "strategy":"max",  "calibration":"isotonic_on_WT",    "blend_weights":{"wt":0.8,"pair":0.2}},
    "TP53":  {"grammar":"context_independent", "strategy":"max",  "calibration":"isotonic_on_WT",    "blend_weights":{"wt":0.8,"pair":0.2}},
    "_fallback": {"grammar":"unknown","strategy":"blend","calibration":"none","blend_weights":{"wt":0.5,"pair":0.5}}
}

# tiny combiner: takes raw WT and PAIR scores (already on 0-1)
def combine_scores(gene: str, wt_score: float, pair_score: float) -> float:
    g = gene.upper()
    cfg = deploy_cfg.get(g, deploy_cfg["_fallback"])
    strat = cfg["strategy"]
    if strat == "pair":   # context-dependent: trust Δ
        return float(pair_score)
    if strat == "max":    # context-independent: gate Δ with WT
        return float(max(wt_score, pair_score))
    if strat == "blend":  # generic convex combo
        w = cfg["blend_weights"]
        return float(w["wt"]*wt_score + w["pair"]*pair_score)
    # default safety
    return float(0.5*wt_score + 0.5*pair_score)

# example usage (pretend scores)
print("EGFR:", combine_scores("EGFR", wt_score=0.61, pair_score=0.78))
print("KRAS:", combine_scores("KRAS", wt_score=0.42, pair_score=0.88))
print("PIK3CA:", combine_scores("PIK3CA", wt_score=0.76, pair_score=0.72))
print("TP53:", combine_scores("TP53", wt_score=0.72, pair_score=0.65))

# persist so inference code can load it
OUT_DIR = "/content/drive/MyDrive/Cancer_gene_database/config"
os.makedirs(OUT_DIR, exist_ok=True)
with open(os.path.join(OUT_DIR, "deploy_config.json"), "w") as f:
    json.dump(deploy_cfg, f, indent=2)
print("Saved:", os.path.join(OUT_DIR, "deploy_config.json"))


deploy config saved, combine() looks sane. next bite: **calibration scaffolding** — fit isotonic for WT-heavy genes and sigmoid (Platt) for $Δ$-heavy genes if we can find out-of-fold (OOF/val) scores; otherwise we fall back to identity.

### Block 3 — find per-sample scores → fit calibrators → save

* Searches for OOF/val score CSVs (common names).

* Fits **Isotonic** (WT) and **Sigmoid/Platt** (PAIR).

* Saves to Drive: `.../config/calibration/*.pkl`.

* If no scores found, prints a clean “identity calibrator” notice and keeps rolling.

In [ ]:
# --- Block 3: per-gene calibration (isotonic for WT, sigmoid for Δ) ---
import os, glob, re, joblib
import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

CFG_DIR   = "/content/drive/MyDrive/Cancer_gene_database/config"
CAL_DIR   = os.path.join(CFG_DIR, "calibration")
os.makedirs(CAL_DIR, exist_ok=True)

DEPLOY_CFG_PATH = os.path.join(CFG_DIR, "deploy_config.json")
SUMMARY_PATH    = "/content/grammar_scan/grammar_type_summary_loco.csv"

deploy_cfg = pd.read_json(DEPLOY_CFG_PATH, typ="series").to_dict()
summary    = pd.read_csv(SUMMARY_PATH)

# ---- 1) Find candidate per-sample score files
ROOTS = [
    "/content/drive/MyDrive/Cancer_gene_database/artifacts",
    "/content/drive/MyDrive",
    "/content",
]
CAND = []
for r in ROOTS:
    if os.path.exists(r):
        CAND += glob.glob(os.path.join(r, "**", "*.csv"), recursive=True)

# Heuristics: file names suggesting oof/val preds
keep = []
name_hints = re.compile(r"(oof|out[_-]?of[_-]?fold|val|valid|dev|pred|scores|logits)", re.I)
for p in CAND:
    fn = os.path.basename(p)
    if name_hints.search(fn) and "metrics" not in fn.lower():
        keep.append(p)
keep = sorted(set(keep))

print(f"[scan] candidate score files: {len(keep)}")
for p in keep[:10]:
    print(" •", p)

# ---- 2) Try loading into a unified frame
def load_scores(p):
    try:
        df = pd.read_csv(p)
        df.columns = [c.strip().lower() for c in df.columns]
        # minimal required columns (try to coalesce)
        # label
        y = None
        for c in ["label","y","y_true","target","truth"]:
            if c in df.columns: y = df[c]; break
        # gene
        g = None
        for c in ["gene","gene_symbol","target_gene","locus_gene","symbol"]:
            if c in df.columns: g = df[c]; break
        # scores
        wt = None; pair = None
        for c in ["wt_score","wt_prob","wt","base_prob","base_score"]:
            if c in df.columns: wt = df[c]; break
        for c in ["pair_score","delta_score","pair_prob","delta_prob","dcnn_prob","ranknet_prob"]:
            if c in df.columns: pair = df[c]; break
        if y is None or g is None or (wt is None and pair is None):
            return None
        out = pd.DataFrame({
            "gene": g.astype(str).str.upper(),
            "y": pd.to_numeric(y, errors="coerce")
        })
        if wt is not None:   out["wt_score"]   = pd.to_numeric(wt, errors="coerce")
        if pair is not None: out["pair_score"] = pd.to_numeric(pair, errors="coerce")
        out["__src"] = p
        return out.dropna(subset=["y"])
    except Exception as e:
        print(f"[skip] {p}: {e}")
        return None

frames = [load_scores(p) for p in keep]
frames = [f for f in frames if f is not None and not f.empty]
scores = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

if scores.empty:
    print("[calibration] No per-sample OOF/val scores found. Using identity calibrators.")
    calibrators = {}  # empty → identity at inference
else:
    # keep our genes only
    genes = set(summary["gene"].tolist())
    scores = scores[scores["gene"].isin(genes)].copy()

    calibrators = {}
    for g in sorted(genes):
        strat = deploy_cfg.get(g, deploy_cfg.get("_fallback", {})).get("strategy", "blend")
        sub = scores[scores["gene"]==g].copy()
        if sub.empty:
            print(f"[calibration] {g}: no rows → identity")
            continue

        # WT calibrator (isotonic) if we will use WT in mix/max
        if strat in {"max","blend"} and "wt_score" in sub.columns:
            try:
                iso = IsotonicRegression(out_of_bounds="clip")
                iso.fit(sub["wt_score"].values, sub["y"].values)
                # evaluate
                yhat = iso.transform(sub["wt_score"].values)
                auc  = roc_auc_score(sub["y"], yhat)
                calibrators[(g, "wt")] = iso
                print(f"[cal] {g} WT isotonic AUC={auc:.3f} on calib set")
            except Exception as e:
                print(f"[cal] {g} WT isotonic failed → identity ({e})")

        # PAIR calibrator (sigmoid/Platt) if we will use pair
        if strat in {"pair","blend","max"} and "pair_score" in sub.columns:
            try:
                # strong C to avoid over-regularizing tiny-n; balanced class_weight
                lr = LogisticRegression(solver="lbfgs", max_iter=200, class_weight="balanced", C=10.0)
                X = sub[["pair_score"]].values
                lr.fit(X, sub["y"].values)
                yhat = lr.predict_proba(X)[:,1]
                auc  = roc_auc_score(sub["y"], yhat)
                calibrators[(g, "pair")] = lr
                print(f"[cal] {g} Δ sigmoid AUC={auc:.3f} on calib set")
            except Exception as e:
                print(f"[cal] {g} Δ sigmoid failed → identity ({e})")

    # persist each calibrator
    for (g, k), model in calibrators.items():
        path = os.path.join(CAL_DIR, f"cal_{g}_{k}.pkl")
        joblib.dump(model, path)
    print(f"[save] wrote {len(calibrators)} calibrators → {CAL_DIR}")

print("[done] calibration scan complete.")


the scanner saw files but didn’t recognize the **label/score** columns. Let’s target the exact files we have listed and auto-map columns.

### Block 3.1 — ingest per-sample preds from known files → fit calibrators
* Loads your **preds** CSVs with a schema-sniffer.

* Extracts `gene`, `y` (label), `wt_score`, `pair_score`.

* Fits **Isotonic** (WT) and **Platt/Sigmoid** (PAIR) per your deploy plan.

* Saves calibrators under `.../config/calibration/`.

In [ ]:
# --- Block 3.1: targeted ingest + calibration (no remount) ---
import os, re, glob, joblib, numpy as np, pandas as pd
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

CFG_DIR = "/content/drive/MyDrive/Cancer_gene_database/config"
CAL_DIR = os.path.join(CFG_DIR, "calibration")
os.makedirs(CAL_DIR, exist_ok=True)

DEPLOY_CFG_PATH = os.path.join(CFG_DIR, "deploy_config.json")
deploy_cfg = pd.read_json(DEPLOY_CFG_PATH, typ="series").to_dict()

# 1) Point directly at your likely per-sample files
TARGET_FILES = [
    "/content/drive/MyDrive/Cancer_gene_database/artifacts/final_receipts_20250816_232227/preds_master.csv",
    "/content/drive/MyDrive/Cancer_gene_database/artifacts/ensemble_twoR_20250816_225822/preds_twoR_mix.csv",
    "/content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_20250816_174000/predictions_loco_gene.csv",
    "/content/drive/MyDrive/Cancer_gene_database/artifacts/cnn_preds_loco_gene.csv",
    "/content/drive/MyDrive/Cancer_gene_database/artifacts/blend_nolearn_20250816_231849/preds_blends.csv",
    "/content/drive/MyDrive/Cancer_gene_database/artifacts/baseline_logreg_tribalanced_preds.csv",
    "/content/drive/MyDrive/Cancer_gene_database/artifacts/baseline_logreg_tribalanced_featlift_preds.csv",
]

TARGET_FILES = [p for p in TARGET_FILES if os.path.exists(p)]
print(f"[targets] using {len(TARGET_FILES)} files")
for p in TARGET_FILES: print(" •", p)

# 2) Column patterns
GENES = {"EGFR","KRAS","PIK3CA","TP53"}

LABEL_CANDS = ["label","y","y_true","target","truth","is_driver","oncogenic"]
GENE_CANDS  = ["gene","gene_symbol","target_gene","locus_gene","symbol"]
WT_CANDS    = ["wt_score","wt_prob","prob_wt","score_wt","wt","base_prob","base_score","wtonly","wt_only"]
PAIR_CANDS  = ["pair_score","pair_prob","prob_pair","score_pair","delta_score","delta_prob",
               "dcnn_prob","cnn_prob","ranknet_prob","twor_prob","pair","delta","dcnn","cnn","twor"]

def pick_col(df, cands):
    cols = [c for c in df.columns if c in cands]
    if cols: return cols[0]
    # fallback: fuzzy contains
    for c in df.columns:
        lc = c.lower()
        if any(k in lc for k in cands):
            return c
    return None

def coerce_label(s):
    s = s.copy()
    if s.dtype == bool:
        return s.astype(int)
    if s.dtype.kind in "if":
        # assume already 0/1 or probabilities—clip to [0,1] and threshold 0.5 only if clearly prob
        u = pd.unique(s.dropna())
        if set(u).issubset({0,1}): return s.astype(int)
        # if looks like prob, threshold at 0.5 for labels
        return (s >= 0.5).astype(int)
    # strings: yes/no/true/false/driver/benign
    map_ = {"true":1,"yes":1,"driver":1,"oncogenic":1,"positive":1,
            "false":0,"no":0,"benign":0,"negative":0,"passenger":0}
    return s.astype(str).str.strip().str.lower().map(map_).astype("Int64").fillna(0).astype(int)

def extract_gene(row):
    g = str(row.get("gene","")).strip().upper()
    if g: return g
    fold = str(row.get("fold",""))
    m = re.search(r"loco[_-]?gene:([A-Za-z0-9]+)", fold, flags=re.I)
    if m: return m.group(1).upper()
    src = str(row.get("__src","")).upper()
    for gg in GENES:
        if gg in src: return gg
    return None

def load_scores_one(path):
    try:
        df = pd.read_csv(path)
    except Exception as e:
        print(f"[skip] {path}: {e}")
        return None
    o = df.copy()
    o.columns = [c.strip().lower() for c in o.columns]
    o["__src"] = path

    # required: label + gene + at least one of wt/pair
    lab_col  = pick_col(o, LABEL_CANDS)
    gene_col = pick_col(o, GENE_CANDS)
    wt_col   = pick_col(o, WT_CANDS)
    pair_col = pick_col(o, PAIR_CANDS)

    # If neither score present, bail
    if wt_col is None and pair_col is None:
        return None

    out = pd.DataFrame()
    if gene_col is not None:
        out["gene"] = o[gene_col].astype(str).str.upper()
    else:
        out["gene"] = o.apply(extract_gene, axis=1)

    if lab_col is not None:
        out["y"] = coerce_label(o[lab_col])
    else:
        # No labels? Can't calibrate from this file.
        return None

    if wt_col is not None:
        out["wt_score"] = pd.to_numeric(o[wt_col], errors="coerce")
    if pair_col is not None:
        out["pair_score"] = pd.to_numeric(o[pair_col], errors="coerce")

    out["__src"] = path
    # keep only our genes and rows with at least one score
    out = out[out["gene"].isin(GENES)]
    keep_mask = out.get("wt_score").notna() if "wt_score" in out.columns else False
    if "pair_score" in out.columns:
        keep_mask = keep_mask | out["pair_score"].notna()
    out = out[keep_mask]
    return out if not out.empty else None

frames = [x for x in (load_scores_one(p) for p in TARGET_FILES) if x is not None]
scores = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

if scores.empty:
    print("[calibration] Still no usable per-sample scores with labels. We can print column headers of each target file if you want to hand-map.")
else:
    print(f"[loaded] rows={len(scores)} across files; by gene:")
    print(scores.groupby("gene").size())

    # 3) Fit calibrators per deploy strategy
    calibrators = {}
    genes = sorted(scores["gene"].unique())
    for g in genes:
        strat = deploy_cfg.get(g, deploy_cfg.get("_fallback", {})).get("strategy","blend")
        sub = scores[scores["gene"]==g].dropna(subset=["y"])
        if sub.empty:
            print(f"[cal] {g}: 0 rows → identity")
            continue

        # WT isotonic if strategy uses WT
        if strat in {"max","blend"} and "wt_score" in sub.columns:
            try:
                iso = IsotonicRegression(out_of_bounds="clip")
                iso.fit(sub["wt_score"].values, sub["y"].values)
                auc = roc_auc_score(sub["y"], iso.transform(sub["wt_score"].values))
                calibrators[(g,"wt")] = iso
                print(f"[cal] {g} WT isotonic AUC={auc:.3f}")
            except Exception as e:
                print(f"[cal] {g} WT isotonic failed → identity ({e})")

        # Pair sigmoid if strategy uses Pair
        if strat in {"pair","blend","max"} and "pair_score" in sub.columns:
            try:
                lr = LogisticRegression(solver="lbfgs", max_iter=300, class_weight="balanced", C=10.0)
                X = sub[["pair_score"]].values
                lr.fit(X, sub["y"].values)
                auc = roc_auc_score(sub["y"], lr.predict_proba(X)[:,1])
                calibrators[(g,"pair")] = lr
                print(f"[cal] {g} Δ sigmoid AUC={auc:.3f}")
            except Exception as e:
                print(f"[cal] {g} Δ sigmoid failed → identity ({e})")

    # 4) Save models
    for (g,k), m in calibrators.items():
        path = os.path.join(CAL_DIR, f"cal_{g}_{k}.pkl")
        joblib.dump(m, path)
    print(f"[save] wrote {len(calibrators)} calibrators → {CAL_DIR}")


calibrators are in the bag. Quick read:

* **WT isotonic** worked where we expected: **PIK3CA (AUC≈0.783), TP53 (AUC≈0.754)** → solid.

* **$Δ$ (pair) Platt** is… meh on tiny genes: EGFR ~0.587, KRAS ~0.518 → small-n + noisy scores. Fine for ranking, don’t worship the raw probs.

* next bite = **wire calibrated inference** so every row gets: wt_cal, pair_cal, and the **final strategy score** per gene.

### Block 4 — calibrated predictor + batch apply (one paste)

This will:

* load your saved `deploy_config.json` + calibrators

* map columns from any preds file

* emit `preds_calibrated.csv` with calibrated + final scores

In [ ]:
# --- Block 4: calibrated predictor + batch apply ---
import os, re, json, joblib
import pandas as pd
from typing import Optional

CFG_DIR = "/content/drive/MyDrive/Cancer_gene_database/config"
CAL_DIR = os.path.join(CFG_DIR, "calibration")

with open(os.path.join(CFG_DIR, "deploy_config.json"), "r") as f:
    DEPLOY = json.load(f)

# load available calibrators
CAL = {}
if os.path.isdir(CAL_DIR):
    for p in os.listdir(CAL_DIR):
        m = re.match(r"cal_([A-Za-z0-9]+)_(wt|pair)\.pkl$", p)
        if m:
            g, k = m.group(1).upper(), m.group(2)
            CAL[(g,k)] = joblib.load(os.path.join(CAL_DIR, p))

def _apply_cal(gene: str, kind: str, score: Optional[float]) -> Optional[float]:
    if score is None:
        return None
    model = CAL.get((gene.upper(), kind))
    if model is None:
        return float(score)
    try:
        if hasattr(model, "transform"):  # IsotonicRegression
            return float(model.transform([score])[0])
        if hasattr(model, "predict_proba"):  # LogisticRegression (Platt)
            return float(model.predict_proba([[score]])[0,1])
    except Exception:
        pass
    return float(score)

def _combine(gene: str, wt_s: Optional[float], pair_s: Optional[float]) -> float:
    g = gene.upper()
    cfg = DEPLOY.get(g, DEPLOY.get("_fallback"))
    strat = cfg["strategy"]
    if strat == "pair":
        return float(pair_s if pair_s is not None else (wt_s if wt_s is not None else 0.5))
    if strat == "max":
        a = wt_s if wt_s is not None else -1
        b = pair_s if pair_s is not None else -1
        return float(max(a, b))
    if strat == "blend":
        w = cfg.get("blend_weights", {"wt":0.5,"pair":0.5})
        a = wt_s if wt_s is not None else 0.0
        b = pair_s if pair_s is not None else 0.0
        return float(w["wt"]*a + w["pair"]*b)
    return float(0.5*((wt_s if wt_s is not None else 0.0) + (pair_s if pair_s is not None else 0.0)))

# column sniffers for arbitrary preds files
LABEL_CANDS = ["label","y","y_true","target","truth","is_driver","oncogenic"]
GENE_CANDS  = ["gene","gene_symbol","target_gene","locus_gene","symbol"]
WT_CANDS    = ["wt_score","wt_prob","prob_wt","score_wt","wt","base_prob","base_score","wtonly","wt_only"]
PAIR_CANDS  = ["pair_score","pair_prob","prob_pair","score_pair","delta_score","delta_prob",
               "dcnn_prob","cnn_prob","ranknet_prob","twor_prob","pair","delta","dcnn","cnn","twor"]

def _pick_col(df, cands):
    cols = [c for c in df.columns if c in cands]
    if cols: return cols[0]
    for c in df.columns:
        lc = c.lower()
        if any(k in lc for k in cands):
            return c
    return None

def calibrate_file(path: str, out_path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df_orig = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]
    gene_col = _pick_col(df, GENE_CANDS)
    wt_col   = _pick_col(df, WT_CANDS)
    pair_col = _pick_col(df, PAIR_CANDS)

    if gene_col is None or (wt_col is None and pair_col is None):
        raise ValueError(f"Need gene + at least one score col. Got columns: {list(df.columns)}")

    # normalize
    out = pd.DataFrame({
        "gene": df[gene_col].astype(str).str.upper(),
    })
    if wt_col:
        out["wt_raw"] = pd.to_numeric(df[wt_col], errors="coerce")
    else:
        out["wt_raw"] = None
    if pair_col:
        out["pair_raw"] = pd.to_numeric(df[pair_col], errors="coerce")
    else:
        out["pair_raw"] = None

    # apply per-gene calibration then strategy
    wt_cal, pair_cal, final = [], [], []
    for g, w, p in out[["gene","wt_raw","pair_raw"]].itertuples(index=False):
        wc = _apply_cal(g, "wt",   None if (w is None or pd.isna(w)) else float(w))
        pc = _apply_cal(g, "pair", None if (p is None or pd.isna(p)) else float(p))
        wt_cal.append(wc)
        pair_cal.append(pc)
        final.append(_combine(g, wc, pc))

    out["wt_cal"] = wt_cal
    out["pair_cal"] = pair_cal
    out["score_final"] = final

    # keep any id columns from original
    keep_ids = [c for c in df_orig.columns if c.lower() in {"gene","pos","hgvs","variant","id","chrom","start","ref","alt"}]
    for c in keep_ids:
        out[c] = df_orig[c]

    out.to_csv(out_path, index=False)
    print(f"[ok] wrote {out_path}  rows={len(out)}")
    return out

# Example: run on your master preds if present
CAND = "/content/drive/MyDrive/Cancer_gene_database/artifacts/final_receipts_20250816_232227/preds_master.csv"
if os.path.exists(CAND):
    _ = calibrate_file(CAND, "/content/drive/MyDrive/Cancer_gene_database/artifacts/final_receipts_20250816_232227/preds_master_calibrated.csv")
else:
    print("[note] set CAND to a preds CSV to produce a calibrated file.")


* EGFR/KRAS: we still **trust $Δ$** for ranking; the Platt fit gives you a **probability-ish** number, but don’t over-interpret it at small-n.

* TP53/PIK3CA: **WT isotonic** is doing real work; then `max(WT,$Δ$)` guards against Δ weirdness.

When that block finishes, we want a want a quick **eval block** that recomputes AUROC/PR **before vs after calibration** from the same preds files (with bootstrap CIs).

### Block 5 — evaluate calibration impact (per gene, pre vs post)

In [ ]:
# --- Block 5: calibration impact eval (AUROC/PR/Brier/ECE) ---
import os, re, json, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

ART_DIR = "/content/drive/MyDrive/Cancer_gene_database/artifacts/final_receipts_20250816_232227"
RAW_PATH = os.path.join(ART_DIR, "preds_master.csv")
CAL_PATH = os.path.join(ART_DIR, "preds_master_calibrated.csv")
CFG_PATH = "/content/drive/MyDrive/Cancer_gene_database/config/deploy_config.json"

assert os.path.exists(RAW_PATH), f"missing {RAW_PATH}"
assert os.path.exists(CAL_PATH), f"missing {CAL_PATH}"

with open(CFG_PATH, "r") as f:
    DEPLOY = json.load(f)

# column sniffers
LABEL_CANDS = ["label","y","y_true","target","truth","is_driver","oncogenic"]
GENE_CANDS  = ["gene","gene_symbol","target_gene","locus_gene","symbol"]
WT_CANDS    = ["wt_score","wt_prob","prob_wt","score_wt","wt","base_prob","base_score","wtonly","wt_only"]
PAIR_CANDS  = ["pair_score","pair_prob","prob_pair","score_pair","delta_score","delta_prob","dcnn_prob","cnn_prob","ranknet_prob","twor_prob","pair","delta","dcnn","cnn","twor"]

def pick_col(df, cands):
    cols = [c for c in df.columns if c in cands]
    if cols: return cols[0]
    for c in df.columns:
        lc = c.lower()
        if any(k in lc for k in cands):
            return c
    return None

def coerce_label(s):
    s = s.copy()
    if s.dtype == bool: return s.astype(int)
    if s.dtype.kind in "if":
        u = pd.unique(s.dropna())
        if set(u).issubset({0,1}): return s.astype(int)
        return (s >= 0.5).astype(int)
    m = {"true":1,"yes":1,"driver":1,"oncogenic":1,"positive":1,
         "false":0,"no":0,"benign":0,"negative":0,"passenger":0}
    return s.astype(str).str.strip().str.lower().map(m).fillna(0).astype(int)

# load raw + calibrated
raw = pd.read_csv(RAW_PATH)
cal = pd.read_csv(CAL_PATH)

raw_cols_lower = {c:c.lower() for c in raw.columns}
raw.columns = [raw_cols_lower[c] for c in raw.columns]
cal.columns = [c.lower() for c in cal.columns]

# align by row order (calibrate_file preserved order)
assert len(raw)==len(cal), "row count mismatch between raw and calibrated"

gene_col  = pick_col(raw, GENE_CANDS); assert gene_col, f"no gene col in {RAW_PATH}"
label_col = pick_col(raw, LABEL_CANDS); assert label_col, f"no label col in {RAW_PATH}"
wt_col    = pick_col(raw, WT_CANDS)
pair_col  = pick_col(raw, PAIR_CANDS)

df = pd.DataFrame({
    "gene": raw[gene_col].astype(str).str.upper(),
    "y": coerce_label(raw[label_col]),
})
if wt_col:   df["wt_raw"]   = pd.to_numeric(raw[wt_col], errors="coerce")
if pair_col: df["pair_raw"] = pd.to_numeric(raw[pair_col], errors="coerce")

# bring calibrated scores
for col in ["wt_cal","pair_cal","score_final"]:
    if col in cal.columns:
        df[col] = pd.to_numeric(cal[col], errors="coerce")
    else:
        df[col] = np.nan

# uncalibrated final using deploy strategy (same logic as before)
def combine(gene, wt, pair):
    cfg = DEPLOY.get(gene, DEPLOY.get("_fallback"))
    strat = cfg["strategy"]
    if strat == "pair":
        return float(pair if pd.notna(pair) else (wt if pd.notna(wt) else 0.5))
    if strat == "max":
        a = wt if pd.notna(wt) else -1
        b = pair if pd.notna(pair) else -1
        return float(max(a, b))
    if strat == "blend":
        w = cfg.get("blend_weights", {"wt":0.5,"pair":0.5})
        a = wt if pd.notna(wt) else 0.0
        b = pair if pd.notna(pair) else 0.0
        return float(w["wt"]*a + w["pair"]*b)
    return float(0.5*((wt if pd.notna(wt) else 0.0)+(pair if pd.notna(pair) else 0.0)))

df["score_pre"] = [combine(g, df.at[i,"wt_raw"] if "wt_raw" in df else np.nan,
                              df.at[i,"pair_raw"] if "pair_raw" in df else np.nan)
                   for i, g in enumerate(df["gene"])]

# metrics helpers
def ece(y, p, bins=10):
    y = np.asarray(y); p = np.asarray(p)
    edges = np.linspace(0,1,bins+1)
    e = 0.0; n = len(y)
    for i in range(bins):
        m = (p>=edges[i]) & (p<edges[i+1] if i<bins-1 else p<=edges[i+1])
        if not m.any(): continue
        conf = p[m].mean()
        acc  = y[m].mean()
        e += (m.sum()/n)*abs(acc-conf)
    return e

def bootstrap_ci(y, p, metric_fn, B=500, seed=42):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(y))
    vals = []
    for _ in range(B):
        s = rng.choice(idx, size=len(idx), replace=True)
        try:
            vals.append(metric_fn(y[s], p[s]))
        except Exception:
            continue
    if not vals:
        return np.nan, np.nan, np.nan
    lo, hi = np.percentile(vals, [2.5,97.5])
    return float(np.mean(vals)), float(lo), float(hi)

def safe_brier(y, p):
    y = np.asarray(y); p = np.asarray(p)
    p = np.clip(p, 1e-6, 1-1e-6)
    return brier_score_loss(y, p)

# per-gene eval
rows=[]
for g, gdf in df.groupby("gene"):
    y = gdf["y"].values
    pre = gdf["score_pre"].values
    post = gdf["score_final"].values  # calibrated final

    # AUROC / PR-AUC / Brier / ECE (with CIs for AUROC & PR)
    au_pre, au_pre_lo, au_pre_hi = bootstrap_ci(y, pre, roc_auc_score)
    au_post, au_post_lo, au_post_hi = bootstrap_ci(y, post, roc_auc_score)
    pr_pre, pr_pre_lo, pr_pre_hi = bootstrap_ci(y, pre, average_precision_score)
    pr_post, pr_post_lo, pr_post_hi = bootstrap_ci(y, post, average_precision_score)

    brier_pre  = safe_brier(y, pre)
    brier_post = safe_brier(y, post)
    ece_pre  = ece(y, pre, bins=10)
    ece_post = ece(y, post, bins=10)

    rows.append({
        "gene": g, "n": len(y),
        "auroc_pre": round(au_pre,4),   "auroc_pre_ci": f"[{au_pre_lo:.3f},{au_pre_hi:.3f}]",
        "auroc_post": round(au_post,4), "auroc_post_ci": f"[{au_post_lo:.3f},{au_post_hi:.3f}]",
        "delta_auroc": round((au_post - au_pre),4),
        "pr_pre": round(pr_pre,4),      "pr_pre_ci": f"[{pr_pre_lo:.3f},{pr_pre_hi:.3f}]",
        "pr_post": round(pr_post,4),    "pr_post_ci": f"[{pr_post_lo:.3f},{pr_post_hi:.3f}]",
        "delta_pr": round((pr_post - pr_pre),4),
        "brier_pre": round(brier_pre,4), "brier_post": round(brier_post,4),
        "delta_brier": round((brier_post - brier_pre),4),
        "ece_pre": round(ece_pre,4),     "ece_post": round(ece_post,4),
        "delta_ece": round((ece_post - ece_pre),4),
    })

eval_df = pd.DataFrame(rows).sort_values("gene")
save_path = os.path.join(ART_DIR, "eval_calibration_summary.csv")
eval_df.to_csv(save_path, index=False)
print("Saved:", save_path)
display(eval_df)


<u>two takeaways from output:</u>

* those **ROC AUC warnings** mean some per-gene slices in `preds_master.csv` **have only one class** (tiny-n reality). AUROC is undefined there; use **PR-AUC, Brier, ECE** instead.

* **TP53 got worse** after calibration because we calibrated components (WT/Δ) and then took `max()`. Taking a max **breaks calibration**. We’ll fix eval first, then (next block) calibrate the final aggregator directly.

let’s keep the “one block at a time” pace. here’s a drop-in eval fix that:

* filters to **LOCO-GENE** if available,

* handles **single-class** slices cleanly (no warnings),

* uses **equal-frequency ECE** (more stable for small n),

* re-runs **pre vs post** metrics.

### Block 5.fix — robust eval (LOCO-only, single-class safe, equal-freq ECE)

In [ ]:
# --- Block 5.fix: robust calibration impact eval ---
import os, re, json, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

ART_DIR = "/content/drive/MyDrive/Cancer_gene_database/artifacts/final_receipts_20250816_232227"
RAW_PATH = os.path.join(ART_DIR, "preds_master.csv")
CAL_PATH = os.path.join(ART_DIR, "preds_master_calibrated.csv")
CFG_PATH = "/content/drive/MyDrive/Cancer_gene_database/config/deploy_config.json"

assert os.path.exists(RAW_PATH), f"missing {RAW_PATH}"
assert os.path.exists(CAL_PATH), f"missing {CAL_PATH}"

with open(CFG_PATH, "r") as f:
    DEPLOY = json.load(f)

# helpers
LABEL_CANDS = ["label","y","y_true","target","truth","is_driver","oncogenic"]
GENE_CANDS  = ["gene","gene_symbol","target_gene","locus_gene","symbol"]
WT_CANDS    = ["wt_score","wt_prob","prob_wt","score_wt","wt","base_prob","base_score","wtonly","wt_only"]
PAIR_CANDS  = ["pair_score","pair_prob","prob_pair","score_pair","delta_score","delta_prob","dcnn_prob","cnn_prob","ranknet_prob","twor_prob","pair","delta","dcnn","cnn","twor"]

def pick_col(df, cands):
    for c in cands:
        if c in df.columns: return c
    # fuzzy
    for c in df.columns:
        if any(k in c.lower() for k in cands):
            return c
    return None

def coerce_label(s):
    if s.dtype == bool: return s.astype(int)
    if s.dtype.kind in "if":
        u = pd.unique(s.dropna())
        if set(u).issubset({0,1}): return s.astype(int)
        return (s >= 0.5).astype(int)
    m = {"true":1,"yes":1,"driver":1,"oncogenic":1,"positive":1,
         "false":0,"no":0,"benign":0,"negative":0,"passenger":0}
    return s.astype(str).str.strip().str.lower().map(m).fillna(0).astype(int)

def combine(gene, wt, pair):
    cfg = DEPLOY.get(gene, DEPLOY.get("_fallback"))
    strat = cfg["strategy"]
    if strat == "pair":  return float(pair if pd.notna(pair) else (wt if pd.notna(wt) else 0.5))
    if strat == "max":   return float(max(wt if pd.notna(wt) else -1, pair if pd.notna(pair) else -1))
    if strat == "blend":
        w = cfg.get("blend_weights", {"wt":0.5,"pair":0.5})
        return float(w["wt"]*(wt if pd.notna(wt) else 0.0) + w["pair"]*(pair if pd.notna(pair) else 0.0))
    return float(0.5*((wt if pd.notna(wt) else 0.0)+(pair if pd.notna(pair) else 0.0)))

def ece_equal_freq(y, p, bins=10):
    y = np.asarray(y); p = np.asarray(p)
    n = len(y)
    if n == 0: return np.nan
    # quantile bins; collapse duplicates
    qs = np.unique(np.quantile(p, np.linspace(0,1,bins+1)))
    e = 0.0
    for i in range(len(qs)-1):
        lo, hi = qs[i], qs[i+1] + (1e-12 if i==len(qs)-2 else 0)  # include right edge on last bin
        m = (p >= lo) & (p < hi)
        if not m.any(): continue
        conf = p[m].mean()
        acc  = y[m].mean()
        e += (m.sum()/n)*abs(acc-conf)
    return float(e)

def safe_metric(y, p, fn):
    # AUROC undefined if one class; return np.nan cleanly
    try:
        return float(fn(y, p))
    except Exception:
        return np.nan

# load raw + calibrated
raw = pd.read_csv(RAW_PATH)
cal = pd.read_csv(CAL_PATH)

# keep LOCO-GENE if fold exists in raw
fold_col = None
for c in raw.columns:
    if c.lower() == "fold":
        fold_col = c
        break
if fold_col is not None:
    loco_mask = raw[fold_col].astype(str).str.contains(r"\bloco[_-]?gene\b", case=False, na=False)
    if not loco_mask.any():
        loco_mask = raw[fold_col].astype(str).str.contains("loco", case=False, na=False)
    raw = raw[loco_mask].reset_index(drop=True)
    cal = cal.loc[raw.index].reset_index(drop=True)

# normalize columns
raw.columns = [c.lower() for c in raw.columns]; cal.columns = [c.lower() for c in cal.columns]
gene_col  = pick_col(raw, GENE_CANDS); assert gene_col, "no gene col"
label_col = pick_col(raw, LABEL_CANDS); assert label_col, "no label col"
wt_col    = pick_col(raw, WT_CANDS)
pair_col  = pick_col(raw, PAIR_CANDS)

df = pd.DataFrame({
    "gene": raw[gene_col].astype(str).str.upper(),
    "y": coerce_label(raw[label_col]),
})
if wt_col:   df["wt_raw"]   = pd.to_numeric(raw[wt_col], errors="coerce")
if pair_col: df["pair_raw"] = pd.to_numeric(raw[pair_col], errors="coerce")
for col in ["wt_cal","pair_cal","score_final"]:
    df[col] = pd.to_numeric(cal[col], errors="coerce") if col in cal.columns else np.nan

df["score_pre"] = [combine(g, df.at[i,"wt_raw"] if "wt_raw" in df else np.nan,
                              df.at[i,"pair_raw"] if "pair_raw" in df else np.nan)
                   for i, g in enumerate(df["gene"])]

# per-gene eval
rows=[]
for g, gdf in df.groupby("gene"):
    y = gdf["y"].values
    pre = gdf["score_pre"].values
    post = gdf["score_final"].values
    n_pos = int((y==1).sum()); n_neg = int((y==0).sum())

    au_pre  = safe_metric(y, pre, roc_auc_score)
    au_post = safe_metric(y, post, roc_auc_score)

    pr_pre  = safe_metric(y, pre, average_precision_score)
    pr_post = safe_metric(y, post, average_precision_score)

    brier_pre  = brier_score_loss(y, np.clip(pre, 1e-6, 1-1e-6))
    brier_post = brier_score_loss(y, np.clip(post,1e-6, 1-1e-6))

    ece_pre  = ece_equal_freq(y, pre, bins=min(10, max(3, len(y)//5)))
    ece_post = ece_equal_freq(y, post, bins=min(10, max(3, len(y)//5)))

    rows.append({
        "gene": g, "n": len(y), "pos": n_pos, "neg": n_neg,
        "auroc_pre": None if np.isnan(au_pre) else round(au_pre,4),
        "auroc_post": None if np.isnan(au_post) else round(au_post,4),
        "delta_auroc": None if np.isnan(au_pre) or np.isnan(au_post) else round(au_post - au_pre,4),
        "pr_pre": None if np.isnan(pr_pre) else round(pr_pre,4),
        "pr_post": None if np.isnan(pr_post) else round(pr_post,4),
        "delta_pr": None if np.isnan(pr_pre) or np.isnan(pr_post) else round(pr_post - pr_pre,4),
        "brier_pre": round(brier_pre,4), "brier_post": round(brier_post,4),
        "delta_brier": round(brier_post - brier_pre,4),
        "ece_pre": round(ece_pre,4), "ece_post": round(ece_post,4),
        "delta_ece": round(ece_post - ece_pre,4),
    })

eval_fix = pd.DataFrame(rows).sort_values("gene")
out_path = os.path.join(ART_DIR, "eval_calibration_summary_LOCO_fix.csv")
eval_fix.to_csv(out_path, index=False)
print("Saved:", out_path)
display(eval_fix)


# Calibration Results — LOCO-GENE (post-calibration audit)

**How to read this:**
- **AUROC** = ranking quality (↑ is better). If a slice has only one class, AUROC is undefined → ignore it.
- **PR-AUC** = ranking under class imbalance (↑).
- **Brier** = probability accuracy / sharpness (↓).
- **ECE** (equal-frequency bins) = calibration gap (↓).  
- We evaluated **LOCO-GENE** only, so no gene leakage.

---

## What happened (numbers = pre → post, $Δ$ = post − pre)

**EGFR (n=39, pos=14, neg=25; strategy = Δ/pair)**  
- **AUROC:** 0.7257 → 0.7257 (**Δ=0.0000**) — ranking unchanged (good; calibration shouldn’t affect pair-only ranking).  
- **PR-AUC:** 0.5711 → 0.5711 (**Δ=0.0000**).  
- **Brier:** **0.3578 → 0.2379 (Δ=−0.1200)** — big probability improvement.  
- **ECE:** **0.5179 → 0.2248 (Δ=−0.2931)** — much better calibrated.  
**Read:** Pair-score Platt smoothing did its job: better probabilities, same ranking. ✅

**KRAS (n=11, pos=8, neg=3; strategy = $Δ$/pair, tiny-n)**  
- **AUROC:** 0.7500 → 0.7500 (**Δ=0.0000**) — stable ranking.  
- **PR-AUC:** 0.9116 → 0.9116 (**Δ=0.0000**).  
- **Brier:** **0.7168 → 0.2640 (Δ=−0.4528)** — massive improvement.  
- **ECE:** **0.9144 → 0.2633 (Δ=−0.6511)** — huge calibration fix.  
**Read:** Tiny sample + extreme raw scores got tamed. Treat the nice Brier/ECE with caution (n=11 is volatile). ⚠️

**PIK3CA (n=24, pos=18, neg=6; strategy = max(WT,$Δ$))**  
- **AUROC:** **0.7593 → 0.8194 (Δ=+0.0602)** — ranking **improved** after component calibration.  
- **PR-AUC:** 0.9212 → 0.9175 (**Δ=−0.0038**) — essentially flat.  
- **Brier:** **0.1918 → 0.1391 (Δ=−0.0527)** — cleaner probabilities.  
- **ECE:** **0.1546 → 0.0214 (Δ=−0.1332)** — excellent calibration.  
**Read:** Here, calibrating WT (isotonic) + $Δ$ (Platt) before the **max** actually helped both calibration and ranking. ✅

**TP53 (n=197, pos=69, neg=128; strategy = max(WT,$Δ$))**  
- **AUROC:** **0.7261 → 0.6246 (Δ=−0.1015)** — ranking got worse.  
- **PR-AUC:** 0.6045 → 0.5739 (**Δ=−0.0307**).  
- **Brier:** **0.2182 → 0.2432 (Δ=+0.0250)** — probabilities worse.  
- **ECE:** **0.1943 → 0.2250 (Δ=+0.0307)** — calibration drifted.  
**Read:** Our warning showed up: calibrating **components** and then taking **max(WT,Δ)** can break calibration and even shuffle winners → ranking drops. ❌

---

## Why TP53 backfired (and PIK3CA didn’t)
Isotonic/Platt are **monotonic** per score, but the **final rule is `max(WT, Δ)`**. Calibrating the two streams independently can flip which stream wins per sample, changing rankings.  
- **PIK3CA:** flips helped (more true positives picked by WT after calibration).  
- **TP53:** flips hurt (we picked the wrong winner more often post-cal).

---

## What’s actually significant here
- **EGFR/KRAS:** Calibration improved **Brier/ECE** without touching AUROC/PR → probabilities are more honest; keep **$Δ$-only with Platt**.  
- **PIK3CA:** Clear across-the-board gains (esp. ECE), plus AUROC bump — current setup is working.  
- **TP53:** Statistically meaningful **degradation** across all metrics — change the plan for this gene.

---

## Playbook adjustments (minimal changes, science-first)
1. **TP53:** **Calibrate the final aggregator, not the components.**  
   - Fit **isotonic on the pre-calibrated final score** `score_pre = max(WT_raw, Δ_raw)` and **use that as the output**.  
   - This preserves ranking (isotonic is monotonic in `score_pre`) and fixes probability drift without cross-stream flip chaos.
2. **PIK3CA:** Keep current **component calibration + max** (it helped). Optionally test final-stage isotonic too; adopt whichever gives lower **Brier/ECE** with no AUROC cost.
3. **EGFR/KRAS:** Keep **pair-only + Platt**; focus improvements elsewhere (features/negatives). Small-n caveat on KRAS stands.

**Sanity notes**
- AUROC being undefined in some slices is expected when a fold has one class; rely on **PR-AUC, Brier, ECE** there.  
- Report all four metrics; highlight **Brier/ECE** for calibration discussions.

**Next step to run later:** add a final-stage isotonic calibrator for **TP53** (and optionally PIK3CA), re-emit `preds_master_finalcal.csv`, and re-run this eval to confirm the fix.


money move right now: **fix TP53** by calibrating the **final aggregator** (`max(WT_raw, Δ_raw`)) with **isotonic** and swapping that in for TP53 only. one bite, one cell:

### --- Block 6: TP53 final-stage isotonic calibration on score_pre(max(WT_raw, Δ_raw)) ---


In [ ]:
# --- Block 6: TP53 final-stage isotonic calibration on score_pre(max(WT_raw, Δ_raw)) ---
import os, re, json, numpy as np, pandas as pd
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

# paths
ART_DIR = "/content/drive/MyDrive/Cancer_gene_database/artifacts/final_receipts_20250816_232227"
RAW_PATH = os.path.join(ART_DIR, "preds_master.csv")                     # raw WT/PAIR + labels
CAL1_PATH = os.path.join(ART_DIR, "preds_master_calibrated.csv")         # component-calibrated ('score_final')
CFG_PATH = "/content/drive/MyDrive/Cancer_gene_database/config/deploy_config.json"
OUT_PATH = os.path.join(ART_DIR, "preds_master_finalcal.csv")            # will write updated file

# load
with open(CFG_PATH, "r") as f:
    DEPLOY = json.load(f)
raw = pd.read_csv(RAW_PATH)
cal1 = pd.read_csv(CAL1_PATH)
raw.columns = [c.lower() for c in raw.columns]; cal1.columns = [c.lower() for c in cal1.columns]

# helpers
LABEL_CANDS = ["label","y","y_true","target","truth","is_driver","oncogenic"]
GENE_CANDS  = ["gene","gene_symbol","target_gene","locus_gene","symbol"]
WT_CANDS    = ["wt_score","wt_prob","prob_wt","score_wt","wt","base_prob","base_score","wtonly","wt_only"]
PAIR_CANDS  = ["pair_score","pair_prob","prob_pair","score_pair","delta_score","delta_prob","dcnn_prob","cnn_prob","ranknet_prob","twor_prob","pair","delta","dcnn","cnn","twor"]

def pick_col(df, cands):
    for c in cands:
        if c in df.columns: return c
    for c in df.columns:
        if any(k in c.lower() for k in cands): return c
    return None

def coerce_label(s):
    if s.dtype == bool: return s.astype(int)
    if s.dtype.kind in "if":
        u = pd.unique(s.dropna())
        if set(u).issubset({0,1}): return s.astype(int)
        return (s >= 0.5).astype(int)
    m = {"true":1,"yes":1,"driver":1,"oncogenic":1,"positive":1,
         "false":0,"no":0,"benign":0,"negative":0,"passenger":0}
    return s.astype(str).str.strip().str.lower().map(m).fillna(0).astype(int)

def combine_pre(gene, wt, pair):
    strat = DEPLOY.get(gene, DEPLOY.get("_fallback"))["strategy"]
    if strat == "pair":  return float(pair if pd.notna(pair) else (wt if pd.notna(wt) else 0.5))
    if strat == "max":   return float(max(wt if pd.notna(wt) else -1, pair if pd.notna(pair) else -1))
    if strat == "blend":
        w = DEPLOY.get(gene, DEPLOY["_fallback"]).get("blend_weights", {"wt":0.5,"pair":0.5})
        return float(w["wt"]*(wt if pd.notna(wt) else 0.0) + w["pair"]*(pair if pd.notna(pair) else 0.0))
    return float(0.5*((wt if pd.notna(wt) else 0.0)+(pair if pd.notna(pair) else 0.0)))

def ece_equal_freq(y, p, bins=10):
    y = np.asarray(y); p = np.asarray(p)
    n = len(y)
    qs = np.unique(np.quantile(p, np.linspace(0,1,max(2, bins)+1)))
    e = 0.0
    for i in range(len(qs)-1):
        lo, hi = qs[i], qs[i+1] + (1e-12 if i==len(qs)-2 else 0)
        m = (p>=lo) & (p<hi)
        if not m.any(): continue
        e += (m.sum()/n)*abs(y[m].mean() - p[m].mean())
    return float(e)

# pick columns
gene_col  = pick_col(raw, GENE_CANDS);   assert gene_col, "no gene column in RAW"
label_col = pick_col(raw, LABEL_CANDS);  assert label_col, "no label column in RAW"
wt_col    = pick_col(raw, WT_CANDS)
pair_col  = pick_col(raw, PAIR_CANDS)

# build working df
df = pd.DataFrame({
    "gene": raw[gene_col].astype(str).str.upper(),
    "y": coerce_label(raw[label_col]),
    "wt_raw":  pd.to_numeric(raw[wt_col], errors="coerce") if wt_col else np.nan,
    "pair_raw":pd.to_numeric(raw[pair_col], errors="coerce") if pair_col else np.nan
})
# LOCO-GENE slice if fold exists
if "fold" in raw.columns:
    loco_mask = raw["fold"].astype(str).str.contains(r"\bloco[_-]?gene\b", case=False, na=False)
    if not loco_mask.any():
        loco_mask = raw["fold"].astype(str).str.contains("loco", case=False, na=False)
    df = df[loco_mask].reset_index(drop=True)
    cal1 = cal1.loc[df.index].reset_index(drop=True)

# pre-combine (before any component calibration)
df["score_pre"] = [combine_pre(g, df.at[i,"wt_raw"], df.at[i,"pair_raw"]) for i, g in enumerate(df["gene"])]

# bring existing component-calibrated final score
if "score_final" in cal1.columns:
    df["score_compcal"] = pd.to_numeric(cal1["score_final"], errors="coerce")
else:
    df["score_compcal"] = df["score_pre"]  # fallback

# --- Fit isotonic on TP53 final-pre, then replace TP53 final ---
mask_tp53 = (df["gene"]=="TP53")
X = df.loc[mask_tp53, "score_pre"].values
y = df.loc[mask_tp53, "y"].values
iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(X, y)
df["score_final_v2"] = df["score_compcal"].copy()
df.loc[mask_tp53, "score_final_v2"] = iso.transform(df.loc[mask_tp53, "score_pre"].values)

# write updated file (keep original columns + new one)
out = cal1.copy()
out["score_final_v2"] = df["score_final_v2"].values
out.to_csv(OUT_PATH, index=False)
print(f"[ok] wrote {OUT_PATH} rows={len(out)}")

# quick TP53 metrics: pre vs comp-cal vs final-iso
def safe_auc(y, p):
    try: return roc_auc_score(y, p)
    except Exception: return np.nan
y_tp = y  # already TP53 subset labels
pre  = df.loc[mask_tp53, "score_pre"].values
post1= df.loc[mask_tp53, "score_compcal"].values
post2= df.loc[mask_tp53, "score_final_v2"].values

def report(name, s):
    return dict(
        name=name,
        auroc=round(float(safe_auc(y_tp, s)),4) if not np.isnan(safe_auc(y_tp, s)) else None,
        pr=round(float(average_precision_score(y_tp, s)),4),
        brier=round(float(brier_score_loss(y_tp, np.clip(s,1e-6,1-1e-6))),4),
        ece=round(float(ece_equal_freq(y_tp, s, bins=min(10, max(3, len(y_tp)//5)))),4)
    )

r_pre  = report("pre=max(WT_raw,Δ_raw)", pre)
r_c1   = report("comp-cal (old score_final)", post1)
r_iso  = report("final-iso (new score_final_v2)", post2)

print("\nTP53 — pre vs comp-cal vs final-iso")
for r in (r_pre, r_c1, r_iso):
    print(r)


TP53 just proved the point — final-stage isotonic fixes the mess. Next best move: make that calibration **honest** with **cross-validated isotonic** (no in-sample glow-up), and save a proper deploy model.

### Block 7 — TP53 CV-isotonic (for eval) + full-fit isotonic (for deploy)

* 5-fold stratified isotonic on score_pre = max(WT_raw, Δ_raw)

* writes:

    * `preds_master_finalcal_cv.csv` with score_final_v3 (TP53 = CV-iso; others unchanged)

    * a **deploy** calibrator `cal_TP53_final_iso.pkl` (fit on all TP53 LOCO for future use)

* prints TP53 metrics: pre vs comp-cal vs **CV-iso** (should look like pre-AUROC ≈ preserved, **Brier/ECE ↓**)

In [ ]:
# --- Block 7: TP53 cross-validated isotonic + deploy calibrator ---
import os, json, numpy as np, pandas as pd
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
import joblib

# paths
ART_DIR  = "/content/drive/MyDrive/Cancer_gene_database/artifacts/final_receipts_20250816_232227"
RAW_PATH = os.path.join(ART_DIR, "preds_master.csv")                 # raw WT/PAIR + labels
CAL1_PATH= os.path.join(ART_DIR, "preds_master_calibrated.csv")      # component-calibrated
OUT_CV   = os.path.join(ART_DIR, "preds_master_finalcal_cv.csv")     # CV-calibrated TP53 final
CFG_PATH = "/content/drive/MyDrive/Cancer_gene_database/config/deploy_config.json"
CAL_DIR  = "/content/drive/MyDrive/Cancer_gene_database/config/calibration"
os.makedirs(CAL_DIR, exist_ok=True)

# helpers
LABEL_CANDS = ["label","y","y_true","target","truth","is_driver","oncogenic"]
GENE_CANDS  = ["gene","gene_symbol","target_gene","locus_gene","symbol"]
WT_CANDS    = ["wt_score","wt_prob","prob_wt","score_wt","wt","base_prob","base_score","wtonly","wt_only"]
PAIR_CANDS  = ["pair_score","pair_prob","prob_pair","score_pair","delta_score","delta_prob","dcnn_prob","cnn_prob","ranknet_prob","twor_prob","pair","delta","dcnn","cnn","twor"]

def pick_col(df, cands):
    for c in cands:
        if c in df.columns: return c
    for c in df.columns:
        if any(k in c.lower() for k in cands): return c
    return None

def coerce_label(s):
    if s.dtype == bool: return s.astype(int)
    if s.dtype.kind in "if":
        u = pd.unique(s.dropna())
        if set(u).issubset({0,1}): return s.astype(int)
        return (s >= 0.5).astype(int)
    m = {"true":1,"yes":1,"driver":1,"oncogenic":1,"positive":1,
         "false":0,"no":0,"benign":0,"negative":0,"passenger":0}
    return s.astype(str).str.strip().str.lower().map(m).fillna(0).astype(int)

def ece_equal_freq(y, p, bins=10):
    y = np.asarray(y); p = np.asarray(p)
    n = len(y)
    qs = np.unique(np.quantile(p, np.linspace(0,1,max(2,bins)+1)))
    e = 0.0
    for i in range(len(qs)-1):
        lo, hi = qs[i], qs[i+1] + (1e-12 if i==len(qs)-2 else 0)
        m = (p>=lo) & (p<hi)
        if not m.any(): continue
        e += (m.sum()/n)*abs(y[m].mean() - p[m].mean())
    return float(e)

# load
with open(CFG_PATH, "r") as f:
    DEPLOY = json.load(f)
raw = pd.read_csv(RAW_PATH)
cal1 = pd.read_csv(CAL1_PATH)
raw.columns = [c.lower() for c in raw.columns]; cal1.columns = [c.lower() for c in cal1.columns]

# pick columns
gene_col  = pick_col(raw, GENE_CANDS);   assert gene_col, "no gene col"
label_col = pick_col(raw, LABEL_CANDS);  assert label_col, "no label col"
wt_col    = pick_col(raw, WT_CANDS)
pair_col  = pick_col(raw, PAIR_CANDS)

# build working df (LOCO-GENE slice if fold provided)
df = pd.DataFrame({
    "gene": raw[gene_col].astype(str).str.upper(),
    "y": coerce_label(raw[label_col]),
    "wt_raw":  pd.to_numeric(raw[wt_col], errors="coerce") if wt_col else np.nan,
    "pair_raw":pd.to_numeric(raw[pair_col], errors="coerce") if pair_col else np.nan
})
if "fold" in raw.columns:
    loco_mask = raw["fold"].astype(str).str.contains(r"\bloco[_-]?gene\b", case=False, na=False)
    if not loco_mask.any():
        loco_mask = raw["fold"].astype(str).str.contains("loco", case=False, na=False)
    df = df[loco_mask].reset_index(drop=True)
    cal1 = cal1.loc[df.index].reset_index(drop=True)

# combine-pre per deploy strategy (we only need TP53 but compute generally)
def combine_pre(gene, wt, pair):
    strat = DEPLOY.get(gene, DEPLOY.get("_fallback"))["strategy"]
    if strat == "pair":  return float(pair if pd.notna(pair) else (wt if pd.notna(wt) else 0.5))
    if strat == "max":   return float(max(wt if pd.notna(wt) else -1, pair if pd.notna(pair) else -1))
    if strat == "blend":
        w = DEPLOY.get(gene, DEPLOY["_fallback"]).get("blend_weights", {"wt":0.5,"pair":0.5})
        return float(w["wt"]*(wt if pd.notna(wt) else 0.0) + w["pair"]*(pair if pd.notna(pair) else 0.0))
    return float(0.5*((wt if pd.notna(wt) else 0.0)+(pair if pd.notna(pair) else 0.0)))

df["score_pre"] = [combine_pre(g, df.at[i,"wt_raw"], df.at[i,"pair_raw"]) for i, g in enumerate(df["gene"])]
df["score_compcal"] = cal1["score_final"].astype(float) if "score_final" in cal1.columns else df["score_pre"]

# ---- CV isotonic for TP53 ----
mask_tp = (df["gene"]=="TP53")
X = df.loc[mask_tp, "score_pre"].values
y = df.loc[mask_tp, "y"].values
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_pred = np.zeros_like(X, dtype=float)
for tr, va in skf.split(X, y):
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(X[tr], y[tr])
    cv_pred[va] = iso.transform(X[va])

# metrics for TP53
def safe_auc(y, p):
    try: return roc_auc_score(y, p)
    except Exception: return np.nan

pre  = X
comp = df.loc[mask_tp, "score_compcal"].values
cv   = cv_pred

r_pre = dict(AUROC=round(float(safe_auc(y, pre)),4) if not np.isnan(safe_auc(y, pre)) else None,
             PR=round(float(average_precision_score(y, pre)),4),
             Brier=round(float(brier_score_loss(y, np.clip(pre,1e-6,1-1e-6))),4),
             ECE=round(float(ece_equal_freq(y, pre, bins=min(10, max(3, len(y)//5)))),4))
r_c1  = dict(AUROC=round(float(safe_auc(y, comp)),4) if not np.isnan(safe_auc(y, comp)) else None,
             PR=round(float(average_precision_score(y, comp)),4),
             Brier=round(float(brier_score_loss(y, np.clip(comp,1e-6,1-1e-6))),4),
             ECE=round(float(ece_equal_freq(y, comp, bins=min(10, max(3, len(y)//5)))),4))
r_cv  = dict(AUROC=round(float(safe_auc(y, cv)),4) if not np.isnan(safe_auc(y, cv)) else None,
             PR=round(float(average_precision_score(y, cv)),4),
             Brier=round(float(brier_score_loss(y, np.clip(cv,1e-6,1-1e-6))),4),
             ECE=round(float(ece_equal_freq(y, cv, bins=min(10, max(3, len(y)//5)))),4))

print("TP53 metrics — pre vs comp-cal vs CV-iso")
print("pre     :", r_pre)
print("comp-cal:", r_c1)
print("CV-iso  :", r_cv)

# write CV-calibrated final column
out = cal1.copy()
out["score_final_v3"] = out.get("score_final", df["score_pre"]).astype(float)  # start from comp-cal or pre
out.loc[mask_tp.index[mask_tp], "score_final_v3"] = cv_pred  # replace TP53 with CV-iso
out.to_csv(OUT_CV, index=False)
print(f"[ok] wrote {OUT_CV} rows={len(out)}")

# ---- Deploy: fit full-data isotonic for TP53 and save model ----
iso_full = IsotonicRegression(out_of_bounds="clip").fit(X, y)
joblib.dump(iso_full, os.path.join(CAL_DIR, "cal_TP53_final_iso.pkl"))
print("[deploy] saved:", os.path.join(CAL_DIR, "cal_TP53_final_iso.pkl"))


Intention now: Block 8 to rerun the per-gene eval using `score_final_v3` (TP53 = CV-iso) and lock the final deploy wrapper to apply `cal_TP53_final_iso.pkl` at inference.

We will do in accord to results above:

* re-evaluate with **TP53’s CV-isotonic final** (`score_final_v3`) so we have clean, final receipts.

### --- Block 8: Final eval using TP53 CV-isotonic (score_final_v3) ---


In [ ]:
# --- Block 8: Final eval using TP53 CV-isotonic (score_final_v3) ---
import os, re, json, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

ART_DIR   = "/content/drive/MyDrive/Cancer_gene_database/artifacts/final_receipts_20250816_232227"
RAW_PATH  = os.path.join(ART_DIR, "preds_master.csv")                 # raw WT/PAIR + labels
COMP_PATH = os.path.join(ART_DIR, "preds_master_calibrated.csv")      # component-calibrated
FINAL_PATH= os.path.join(ART_DIR, "preds_master_finalcal_cv.csv")     # TP53 CV-iso in score_final_v3
CFG_PATH  = "/content/drive/MyDrive/Cancer_gene_database/config/deploy_config.json"

assert os.path.exists(RAW_PATH), RAW_PATH
assert os.path.exists(COMP_PATH), COMP_PATH
assert os.path.exists(FINAL_PATH), FINAL_PATH

with open(CFG_PATH, "r") as f:
    DEPLOY = json.load(f)

# helpers
LABEL_CANDS = ["label","y","y_true","target","truth","is_driver","oncogenic"]
GENE_CANDS  = ["gene","gene_symbol","target_gene","locus_gene","symbol"]
WT_CANDS    = ["wt_score","wt_prob","prob_wt","score_wt","wt","base_prob","base_score","wtonly","wt_only"]
PAIR_CANDS  = ["pair_score","pair_prob","prob_pair","score_pair","delta_score","delta_prob","dcnn_prob","cnn_prob","ranknet_prob","twor_prob","pair","delta","dcnn","cnn","twor"]

def pick_col(df, cands):
    for c in cands:
        if c in df.columns: return c
    for c in df.columns:
        if any(k in c.lower() for k in cands): return c
    return None

def coerce_label(s):
    if s.dtype == bool: return s.astype(int)
    if s.dtype.kind in "if":
        u = pd.unique(s.dropna())
        if set(u).issubset({0,1}): return s.astype(int)
        return (s >= 0.5).astype(int)
    m = {"true":1,"yes":1,"driver":1,"oncogenic":1,"positive":1,
         "false":0,"no":0,"benign":0,"negative":0,"passenger":0}
    return s.astype(str).str.strip().str.lower().map(m).fillna(0).astype(int)

def combine_pre(gene, wt, pair):
    cfg = DEPLOY.get(gene, DEPLOY.get("_fallback"))
    strat = cfg["strategy"]
    if strat == "pair":  return float(pair if pd.notna(pair) else (wt if pd.notna(wt) else 0.5))
    if strat == "max":   return float(max(wt if pd.notna(wt) else -1, pair if pd.notna(pair) else -1))
    if strat == "blend":
        w = cfg.get("blend_weights", {"wt":0.5,"pair":0.5})
        return float(w["wt"]*(wt if pd.notna(wt) else 0.0) + w["pair"]*(pair if pd.notna(pair) else 0.0))
    return float(0.5*((wt if pd.notna(wt) else 0.0)+(pair if pd.notna(pair) else 0.0)))

def ece_equal_freq(y, p, bins=10):
    y = np.asarray(y); p = np.asarray(p)
    n = len(y)
    qs = np.unique(np.quantile(p, np.linspace(0,1,max(2,bins)+1)))
    e = 0.0
    for i in range(len(qs)-1):
        lo, hi = qs[i], qs[i+1] + (1e-12 if i==len(qs)-2 else 0)
        m = (p>=lo) & (p<hi)
        if not m.any(): continue
        e += (m.sum()/n)*abs(y[m].mean() - p[m].mean())
    return float(e)

def safe_metric(y, p, fn):
    try: return float(fn(y, p))
    except Exception: return np.nan

# load files
raw   = pd.read_csv(RAW_PATH);   raw.columns   = [c.lower() for c in raw.columns]
comp  = pd.read_csv(COMP_PATH);  comp.columns  = [c.lower() for c in comp.columns]
final = pd.read_csv(FINAL_PATH); final.columns = [c.lower() for c in final.columns]

# LOCO-GENE slice if available (align all three by the same indices)
if "fold" in raw.columns:
    loco = raw["fold"].astype(str).str.contains(r"\bloco[_-]?gene\b", case=False, na=False)
    if not loco.any():
        loco = raw["fold"].astype(str).str.contains("loco", case=False, na=False)
    idx = raw[loco].index
    raw   = raw.loc[idx].reset_index(drop=True)
    comp  = comp.loc[idx].reset_index(drop=True)
    final = final.loc[idx].reset_index(drop=True)

gene_col  = pick_col(raw, GENE_CANDS);  assert gene_col
label_col = pick_col(raw, LABEL_CANDS); assert label_col
wt_col    = pick_col(raw, WT_CANDS)
pair_col  = pick_col(raw, PAIR_CANDS)

df = pd.DataFrame({
    "gene": raw[gene_col].astype(str).str.upper(),
    "y": coerce_label(raw[label_col]),
})
if wt_col:   df["wt_raw"]   = pd.to_numeric(raw[wt_col], errors="coerce")
if pair_col: df["pair_raw"] = pd.to_numeric(raw[pair_col], errors="coerce")

df["score_pre"]   = [combine_pre(g, df.at[i,"wt_raw"] if "wt_raw" in df else np.nan,
                                    df.at[i,"pair_raw"] if "pair_raw" in df else np.nan)
                     for i, g in enumerate(df["gene"])]
df["score_comp"]  = pd.to_numeric(comp.get("score_final", df["score_pre"]), errors="coerce")
df["score_final"] = pd.to_numeric(final.get("score_final_v3", df["score_comp"]), errors="coerce")

rows=[]
for g, gdf in df.groupby("gene"):
    y = gdf["y"].values
    pre   = gdf["score_pre"].values
    compc = gdf["score_comp"].values
    fin   = gdf["score_final"].values
    n_pos = int((y==1).sum()); n_neg = int((y==0).sum())

    au_pre  = safe_metric(y, pre, roc_auc_score)
    au_comp = safe_metric(y, compc, roc_auc_score)
    au_fin  = safe_metric(y, fin, roc_auc_score)

    pr_pre  = safe_metric(y, pre,  average_precision_score)
    pr_comp = safe_metric(y, compc,average_precision_score)
    pr_fin  = safe_metric(y, fin,  average_precision_score)

    brier_pre  = brier_score_loss(y, np.clip(pre,  1e-6, 1-1e-6))
    brier_comp = brier_score_loss(y, np.clip(compc,1e-6, 1-1e-6))
    brier_fin  = brier_score_loss(y, np.clip(fin,  1e-6, 1-1e-6))

    ece_pre  = ece_equal_freq(y, pre,  bins=min(10, max(3, len(y)//5)))
    ece_comp = ece_equal_freq(y, compc,bins=min(10, max(3, len(y)//5)))
    ece_fin  = ece_equal_freq(y, fin,  bins=min(10, max(3, len(y)//5)))

    rows.append({
        "gene": g, "n": len(y), "pos": n_pos, "neg": n_neg,
        "AUROC_pre": None if np.isnan(au_pre) else round(au_pre,4),
        "AUROC_comp": None if np.isnan(au_comp) else round(au_comp,4),
        "AUROC_final": None if np.isnan(au_fin) else round(au_fin,4),
        "ΔAUROC(comp-pre)": None if (np.isnan(au_pre) or np.isnan(au_comp)) else round(au_comp-au_pre,4),
        "ΔAUROC(final-pre)": None if (np.isnan(au_pre) or np.isnan(au_fin))  else round(au_fin-au_pre,4),
        "PR_pre": None if np.isnan(pr_pre) else round(pr_pre,4),
        "PR_comp": None if np.isnan(pr_comp) else round(pr_comp,4),
        "PR_final": None if np.isnan(pr_fin) else round(pr_fin,4),
        "Brier_pre": round(brier_pre,4), "Brier_comp": round(brier_comp,4), "Brier_final": round(brier_fin,4),
        "ECE_pre": round(ece_pre,4), "ECE_comp": round(ece_comp,4), "ECE_final": round(ece_fin,4),
    })

eval_final = pd.DataFrame(rows).sort_values("gene")
out_path = os.path.join(ART_DIR, "eval_calibration_summary_LOCO_FINAL.csv")
eval_final.to_csv(out_path, index=False)
print("Saved:", out_path)
display(eval_final)


results say: keep **EGFR/KRAS → Δ-only+Platt, PIK3CA → component-cal + max, TP53 → final-stage isotonic on** `max(WT,Δ)`. Next up is a clean **deploy predictor** that bakes those rules in.

### --- Block 9: Final deploy predictor (gene-aware, with TP53 final-stage isotonic) ---


In [ ]:
# --- Block 9: Final deploy predictor (gene-aware, with TP53 final-stage isotonic) ---
import os, re, json, joblib, numpy as np, pandas as pd
from typing import Optional

# paths
ROOT_CFG = "/content/drive/MyDrive/Cancer_gene_database/config"
CAL_DIR  = os.path.join(ROOT_CFG, "calibration")
DEPLOY_CFG_PATH = os.path.join(ROOT_CFG, "deploy_config.json")

# load deploy plan
with open(DEPLOY_CFG_PATH, "r") as f:
    DEPLOY = json.load(f)

# load component calibrators (WT isotonic, Δ/PAIR Platt) if available
CAL = {}
if os.path.isdir(CAL_DIR):
    for p in os.listdir(CAL_DIR):
        m = re.match(r"cal_([A-Za-z0-9]+)_(wt|pair)\.pkl$", p)
        if m:
            g, k = m.group(1).upper(), m.group(2)
            CAL[(g,k)] = joblib.load(os.path.join(CAL_DIR, p))

# load TP53 final-stage isotonic (fitted in Block 7) if present
CAL_TP53_FINAL = None
tp53_final_path = os.path.join(CAL_DIR, "cal_TP53_final_iso.pkl")
if os.path.exists(tp53_final_path):
    CAL_TP53_FINAL = joblib.load(tp53_final_path)

# --- helpers to sniff columns in arbitrary preds files
LABEL_CANDS = ["label","y","y_true","target","truth","is_driver","oncogenic"]
GENE_CANDS  = ["gene","gene_symbol","target_gene","locus_gene","symbol"]
WT_CANDS    = ["wt_score","wt_prob","prob_wt","score_wt","wt","base_prob","base_score","wtonly","wt_only"]
PAIR_CANDS  = ["pair_score","pair_prob","prob_pair","score_pair","delta_score","delta_prob",
               "dcnn_prob","cnn_prob","ranknet_prob","twor_prob","pair","delta","dcnn","cnn","twor"]

def _pick_col(df, cands):
    for c in cands:
        if c in df.columns: return c
    for c in df.columns:
        if any(k in c.lower() for k in cands): return c
    return None

# --- core scoring primitives
def _apply_component_cal(gene: str, kind: str, score: Optional[float]) -> Optional[float]:
    """Apply WT isotonic or Δ Platt if present; identity otherwise."""
    if score is None or (isinstance(score, float) and (np.isnan(score))):
        return None
    model = CAL.get((gene.upper(), kind))
    if model is None:
        return float(score)
    try:
        # IsotonicRegression has .transform; LogisticRegression has .predict_proba
        if hasattr(model, "transform"):
            return float(model.transform([float(score)])[0])
        if hasattr(model, "predict_proba"):
            return float(model.predict_proba([[float(score)]])[0,1])
    except Exception:
        return float(score)
    return float(score)

def _final_tp53(score_pre: float) -> float:
    """TP53 final-stage isotonic applied to score_pre = max(WT_raw, Δ_raw)."""
    if CAL_TP53_FINAL is None or score_pre is None or np.isnan(score_pre):
        return float(score_pre) if score_pre is not None else 0.5
    try:
        return float(CAL_TP53_FINAL.transform([float(score_pre)])[0])
    except Exception:
        return float(score_pre)

def predict_proba(gene: str, wt_raw: Optional[float], pair_raw: Optional[float]) -> float:
    """Gene-aware final probability following our receipts."""
    g = gene.upper()
    cfg = DEPLOY.get(g, DEPLOY.get("_fallback"))
    strat = cfg["strategy"]

    # SPECIAL CASE — TP53: use final-stage isotonic on pre-score; skip component cals
    if g == "TP53":
        pre = max(wt_raw if wt_raw is not None else -1, pair_raw if pair_raw is not None else -1)
        return _final_tp53(pre)

    # EGFR/KRAS: Δ-only with Platt (component-level)
    if g in {"EGFR","KRAS"}:
        pair_c = _apply_component_cal(g, "pair", pair_raw)
        # fallback if pair missing
        if pair_c is None and wt_raw is not None:
            return float(wt_raw)
        return float(pair_c if pair_c is not None else 0.5)

    # PIK3CA (and any other context-independent gene): component-cal then max
    if g == "PIK3CA" or strat == "max":
        wt_c   = _apply_component_cal(g, "wt",   wt_raw)
        pair_c = _apply_component_cal(g, "pair", pair_raw)
        # if one missing, use the other
        if wt_c is None and pair_c is None:
            return 0.5
        if wt_c is None: return float(pair_c)
        if pair_c is None: return float(wt_c)
        return float(max(wt_c, pair_c))

    # generic fallback: blend with any available component calibrators
    if strat == "blend":
        w = cfg.get("blend_weights", {"wt":0.5,"pair":0.5})
        wt_c   = _apply_component_cal(g, "wt",   wt_raw) or 0.0
        pair_c = _apply_component_cal(g, "pair", pair_raw) or 0.0
        return float(w["wt"]*wt_c + w["pair"]*pair_c)

    # strat == "pair" or unknown: prefer pair (calibrated), fallback to wt
    pair_c = _apply_component_cal(g, "pair", pair_raw)
    if pair_c is not None: return float(pair_c)
    return float(_apply_component_cal(g, "wt", wt_raw) or 0.5)

# --- batch utility: read a preds CSV, produce a final 'proba' column
def score_file(in_path: str, out_path: str, use_loco_only: bool = True) -> pd.DataFrame:
    df0 = pd.read_csv(in_path)
    df  = df0.copy()
    df.columns = [c.lower() for c in df.columns]

    gene_col = _pick_col(df, GENE_CANDS);  assert gene_col, "need a gene column"
    wt_col   = _pick_col(df, WT_CANDS)
    pair_col = _pick_col(df, PAIR_CANDS)

    if use_loco_only and "fold" in df.columns:
        mask = df["fold"].astype(str).str.contains(r"\bloco[_-]?gene\b", case=False, na=False)
        if not mask.any():
            mask = df["fold"].astype(str).str.contains("loco", case=False, na=False)
        df = df[mask].copy()

    genes = df[gene_col].astype(str).str.upper().tolist()
    wt    = pd.to_numeric(df[wt_col], errors="coerce").tolist() if wt_col else [None]*len(df)
    pair  = pd.to_numeric(df[pair_col], errors="coerce").tolist() if pair_col else [None]*len(df)

    finals = [predict_proba(g, w, p) for g, w, p in zip(genes, wt, pair)]
    df_out = df0.loc[df.index].copy()  # keep original cols/rows order for selected slice
    df_out["proba_deploy"] = finals
    df_out.to_csv(out_path, index=False)
    print(f"[deploy] wrote {out_path} rows={len(df_out)}")
    return df_out

# --- quick smoke test on your master preds (writes alongside prior outputs)
CAND = "/content/drive/MyDrive/Cancer_gene_database/artifacts/final_receipts_20250816_232227/preds_master.csv"
if os.path.exists(CAND):
    _ = score_file(CAND, os.path.join(os.path.dirname(CAND), "preds_master_DEPLOY.csv"), use_loco_only=True)
else:
    print("[note] set CAND to a preds CSV to generate deploy probabilities.")


next bite is **operating points** so deploy isn’t just “nice probs” — it makes yes/no calls per gene. we’ll compute three candidates per gene (F1-opt, Youden’s J, and “recall≥0.80 then best precision”), score them, and save a clean `thresholds.json` + `operating_points.csv`.

### --- Block 10: Per-gene operating thresholds (F1-opt, Youden J, Recall≥0.80) ---


In [ ]:
# --- Block 10: Per-gene operating thresholds (F1-opt, Youden J, Recall≥0.80) ---
import os, json, numpy as np, pandas as pd
from sklearn.metrics import (
    precision_recall_curve, roc_curve, roc_auc_score,
    average_precision_score, confusion_matrix, brier_score_loss
)

ART_DIR   = "/content/drive/MyDrive/Cancer_gene_database/artifacts/final_receipts_20250816_232227"
RAW_PATH  = os.path.join(ART_DIR, "preds_master.csv")                 # has labels + gene + fold
DEPLOY_PATH = os.path.join(ART_DIR, "preds_master_DEPLOY.csv")        # has proba_deploy
OUT_THRESH_JSON = os.path.join(ART_DIR, "thresholds.json")
OUT_OPTS_CSV    = os.path.join(ART_DIR, "operating_points.csv")

assert os.path.exists(RAW_PATH), RAW_PATH
assert os.path.exists(DEPLOY_PATH), DEPLOY_PATH

# --- load & align (LOCO-GENE only) ---
raw = pd.read_csv(RAW_PATH); raw.columns = [c.lower() for c in raw.columns]
dep = pd.read_csv(DEPLOY_PATH); dep.columns = [c.lower() for c in dep.columns]

# Keep LOCO-GENE slice and row alignment
if "fold" in raw.columns:
    loco = raw["fold"].astype(str).str.contains(r"\bloco[_-]?gene\b", case=False, na=False)
    if not loco.any():
        loco = raw["fold"].astype(str).str.contains("loco", case=False, na=False)
    raw = raw[loco].reset_index(drop=True)
    dep  = dep.loc[raw.index].reset_index(drop=True)

# Column sniffers
def pick_col(df, cands):
    for c in cands:
        if c in df.columns: return c
    for c in df.columns:
        if any(k in c.lower() for k in cands): return c
    return None

LABEL_CANDS = ["label","y","y_true","target","truth","is_driver","oncogenic"]
GENE_CANDS  = ["gene","gene_symbol","target_gene","locus_gene","symbol"]

gene_col  = pick_col(raw, GENE_CANDS);  assert gene_col, "no gene column in RAW"
label_col = pick_col(raw, LABEL_CANDS); assert label_col, "no label column in RAW"
proba_col = "proba_deploy";             assert proba_col in dep.columns, "DEPLOY lacks proba_deploy"

def coerce_label(s):
    if s.dtype == bool: return s.astype(int)
    if s.dtype.kind in "if":
        u = pd.unique(s.dropna());
        if set(u).issubset({0,1}): return s.astype(int)
        return (s >= 0.5).astype(int)
    m = {"true":1,"yes":1,"driver":1,"oncogenic":1,"positive":1,
         "false":0,"no":0,"benign":0,"negative":0,"passenger":0}
    return s.astype(str).str.strip().str.lower().map(m).fillna(0).astype(int)

df = pd.DataFrame({
    "gene": raw[gene_col].astype(str).str.upper(),
    "y": coerce_label(raw[label_col]),
    "p": pd.to_numeric(dep[proba_col], errors="coerce").clip(0,1)
})

# --- helpers ---
def safe_auc(y, p):
    try: return roc_auc_score(y, p)
    except Exception: return np.nan

def pr_f1_threshold(y, p):
    # returns threshold maximizing F1 (computed over PR curve)
    prec, rec, thr = precision_recall_curve(y, p)
    # precision_recall_curve returns len(thr)+1 points for prec/rec
    f1 = np.where((prec+rec)>0, 2*prec*rec/(prec+rec), 0)
    idx = int(np.nanargmax(f1))
    # map back to threshold: last point has no threshold; guard that case
    t = thr[idx] if idx < len(thr) else 0.5
    return float(t), float(f1[idx]), float(prec[idx]), float(rec[idx])

def youden_j_threshold(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    j = tpr - fpr
    idx = int(np.nanargmax(j))
    return float(thr[idx]), float(j[idx]), float(tpr[idx]), float(1-fpr[idx])

def recall_target_threshold(y, p, target=0.80):
    prec, rec, thr = precision_recall_curve(y, p)
    # find candidates with recall >= target, choose highest precision (ties → higher recall)
    cand = np.where(rec >= target)[0]
    if cand.size == 0:
        # fallback: take F1-opt if target unreachable
        t, f1, pr, rc = pr_f1_threshold(y, p)
        return float(t), float(pr), float(rc), True
    # precision_recall_curve ordering is descending thresholds; last point no threshold
    pr_slice = prec[cand]; rc_slice = rec[cand]
    best = int(np.nanargmax(pr_slice))
    idx = cand[best]
    # threshold index mapping
    t = thr[idx] if idx < len(thr) else 0.5
    return float(t), float(pr_slice[best]), float(rc_slice[best]), False

def score_at(y, p, th):
    yhat = (p >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0,1]).ravel()
    acc = (tp+tn)/(tp+tn+fp+fn) if (tp+tn+fp+fn)>0 else np.nan
    prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
    rec  = tp/(tp+fn) if (tp+fn)>0 else 0.0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
    bacc = 0.5*((tp/(tp+fn) if (tp+fn)>0 else 0.0) + (tn/(tn+fp) if (tn+fp)>0 else 0.0))
    brier = brier_score_loss(y, np.clip(p,1e-6,1-1e-6))
    return {"tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp),
            "acc":round(acc,4),"f1":round(f1,4),"prec":round(prec,4),"rec":round(rec,4),
            "bacc":round(bacc,4),"brier":round(brier,4)}

# --- compute per gene ---
rows = []
th_json = {}
for g, gdf in df.groupby("gene"):
    y = gdf["y"].values
    p = gdf["p"].values
    if len(np.unique(y)) < 2:
        # single-class slice: set neutral threshold 0.5 and report only counts
        th = 0.5
        m = score_at(y, p, th)
        rows.append({"gene": g, "metric": "single_class", "threshold": th, **m,
                     "auroc": np.nan, "pr_auc": average_precision_score(y, p)})
        th_json[g] = {"recommended": th, "basis": "single_class", "alternates": {}}
        continue

    # global curves
    auroc = safe_auc(y, p)
    prauc = average_precision_score(y, p)

    # candidates
    t_f1, f1_best, f1_prec, f1_rec = pr_f1_threshold(y, p)
    t_j, j_best, tpr, tnr = youden_j_threshold(y, p)
    t_r, pr_at, rc_at, fell_back = recall_target_threshold(y, p, target=0.80)

    # score them
    m_f1 = score_at(y, p, t_f1); m_f1.update({"auroc": round(float(auroc),4), "pr_auc": round(float(prauc),4)})
    m_j  = score_at(y, p, t_j);  m_j.update({"auroc": round(float(auroc),4), "pr_auc": round(float(prauc),4)})
    m_rt = score_at(y, p, t_r);  m_rt.update({"auroc": round(float(auroc),4), "pr_auc": round(float(prauc),4)})

    rows += [
        {"gene": g, "metric": "F1_opt", "threshold": round(float(t_f1),6), **m_f1},
        {"gene": g, "metric": "YoudenJ", "threshold": round(float(t_j),6), **m_j},
        {"gene": g, "metric": ("Recall>=0.80" + ("_fallbackF1" if fell_back else "")),
         "threshold": round(float(t_r),6), **m_rt},
    ]

    # recommended policy: context-dependent (EGFR/KRAS) → YoudenJ; context-independent (PIK3CA/TP53) → Recall≥0.80
    if g in {"EGFR","KRAS"}:
        rec_policy = "YoudenJ"; rec_th = float(t_j)
    else:
        rec_policy = "Recall>=0.80" + ("_fallbackF1" if fell_back else ""); rec_th = float(t_r)

    th_json[g] = {
        "recommended": rec_th,
        "policy": rec_policy,
        "alternates": {"F1_opt": float(t_f1), "YoudenJ": float(t_j), "Recall>=0.80": float(t_r)}
    }

# --- write artifacts ---
op_df = pd.DataFrame(rows).sort_values(["gene","metric"])
op_df.to_csv(OUT_OPTS_CSV, index=False)
with open(OUT_THRESH_JSON, "w") as f:
    json.dump(th_json, f, indent=2)

print(f"[ok] wrote {OUT_OPTS_CSV}")
print(f"[ok] wrote {OUT_THRESH_JSON}")

# show quick summary table (recommended only)
rec_rows = []
for g, cfg in th_json.items():
    r = op_df[(op_df["gene"]==g) & (op_df["threshold"].round(6)==round(cfg["recommended"],6))]
    if not r.empty:
        rec_rows.append(r.iloc[0].to_dict())
rec_df = pd.DataFrame(rec_rows)[["gene","metric","threshold","f1","prec","rec","bacc","brier","auroc","pr_auc","tp","fp","fn","tn"]]
display(rec_df.sort_values("gene").reset_index(drop=True))


**Rock solid.**
 We'll lock this in as “deploy-ready”: apply the per-gene thresholds to the calibrated probs and spit out final calls + a clean confusion summary.

### Block 11 — make the actual calls (per-gene thresholds) + summary

In [ ]:
# --- Block 11: Apply thresholds → final calls + summary receipts ---
import os, json, numpy as np, pandas as pd
from sklearn.metrics import confusion_matrix, roc_auc_score, average_precision_score, brier_score_loss

ART_DIR = "/content/drive/MyDrive/Cancer_gene_database/artifacts/final_receipts_20250816_232227"
DEPLOY_PATH = os.path.join(ART_DIR, "preds_master_DEPLOY.csv")      # has proba_deploy
RAW_PATH    = os.path.join(ART_DIR, "preds_master.csv")             # has labels + gene + (fold)
TH_PATH     = os.path.join(ART_DIR, "thresholds.json")

OUT_CALLS   = os.path.join(ART_DIR, "preds_master_DEPLOY_called.csv")
OUT_SUMMARY = os.path.join(ART_DIR, "eval_operating_points_summary.csv")

assert os.path.exists(DEPLOY_PATH), DEPLOY_PATH
assert os.path.exists(RAW_PATH), RAW_PATH
assert os.path.exists(TH_PATH), TH_PATH

with open(TH_PATH, "r") as f:
    TH = json.load(f)

# load + align LOCO-GENE rows
raw = pd.read_csv(RAW_PATH); raw.columns = [c.lower() for c in raw.columns]
dep = pd.read_csv(DEPLOY_PATH); dep.columns = [c.lower() for c in dep.columns]

if "fold" in raw.columns:
    loco = raw["fold"].astype(str).str.contains(r"\bloco[_-]?gene\b", case=False, na=False)
    if not loco.any():
        loco = raw["fold"].astype(str).str.contains("loco", case=False, na=False)
    raw = raw[loco].reset_index(drop=True)
    dep  = dep.loc[raw.index].reset_index(drop=True)

# column sniffers
def pick_col(df, cands):
    for c in cands:
        if c in df.columns: return c
    for c in df.columns:
        if any(k in c.lower() for k in cands): return c
    return None

LABEL_CANDS = ["label","y","y_true","target","truth","is_driver","oncogenic"]
GENE_CANDS  = ["gene","gene_symbol","target_gene","locus_gene","symbol"]
gene_col  = pick_col(raw, GENE_CANDS);  assert gene_col, "no gene column"
label_col = pick_col(raw, LABEL_CANDS); assert label_col, "no label column"
assert "proba_deploy" in dep.columns, "DEPLOY file is missing proba_deploy"

# build working df
df = pd.DataFrame({
    "gene": raw[gene_col].astype(str).str.upper(),
    "y": (raw[label_col].astype(str).str.strip().str.lower()
                    .map({"true":1,"yes":1,"driver":1,"oncogenic":1,"positive":1,
                          "false":0,"no":0,"benign":0,"negative":0,"passenger":0})
                    .fillna(raw[label_col]).astype(float).round().astype(int)),
    "p": pd.to_numeric(dep["proba_deploy"], errors="coerce").clip(0,1),
})

# apply per-gene thresholds
def thr_for_gene(g):
    info = TH.get(g, None)
    if isinstance(info, dict) and "recommended" in info:
        return float(info["recommended"]), info.get("policy","custom")
    # default fallback
    return 0.5, "default_0.5"

calls, used_thr, policy = [], [], []
for g, p in zip(df["gene"], df["p"]):
    t, pol = thr_for_gene(g)
    calls.append(int(p >= t))
    used_thr.append(t)
    policy.append(pol)

df_out = dep.copy()
df_out["gene"] = df["gene"]
df_out["label"] = df["y"]
df_out["proba_deploy"] = df["p"]
df_out["threshold_used"] = used_thr
df_out["policy_used"] = policy
df_out["call"] = calls
df_out.to_csv(OUT_CALLS, index=False)
print(f"[ok] wrote {OUT_CALLS}  rows={len(df_out)}")

# per-gene confusion + metrics at the chosen threshold
rows=[]
for g, gdf in df.groupby("gene"):
    y = gdf["y"].values
    p = gdf["p"].values
    t, pol = thr_for_gene(g)
    yhat = (p >= t).astype(int)

    # confusion
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0,1]).ravel()
    # metrics
    prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
    rec  = tp/(tp+fn) if (tp+fn)>0 else 0.0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
    bacc = 0.5*((tp/(tp+fn) if (tp+fn)>0 else 0.0) + (tn/(tn+fp) if (tn+fp)>0 else 0.0))
    # ranking metrics (safe)
    try: au = roc_auc_score(y, p)
    except Exception: au = np.nan
    prauc = average_precision_score(y, p)
    brier = brier_score_loss(y, np.clip(p,1e-6,1-1e-6))

    rows.append({
        "gene": g, "n": int(len(y)), "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
        "threshold": round(float(t),6), "policy": pol,
        "precision": round(prec,4), "recall": round(rec,4), "f1": round(f1,4),
        "bacc": round(bacc,4), "brier": round(brier,4),
        "auroc": None if np.isnan(au) else round(au,4), "pr_auc": round(prauc,4),
    })

summary = pd.DataFrame(rows).sort_values("gene")
summary.to_csv(OUT_SUMMARY, index=False)
print(f"[ok] wrote {OUT_SUMMARY}")
display(summary)


* **EGFR/KRAS**: thresholds look tight; good balance (F1 ≈ .69/.88).

* **PIK3CA**: recall=precision=.83 — chef’s kiss.

* **TP53**: We cranked recall to 1.00 (no FNs), but precision is 0.37 (lots of FPs). That might be fine if TP53 is a “screen then verify” stage… but if we want fewer false alarms, we should tune TP53’s threshold to a recall floor we can live with (say 0.90) and grab the best precision under that constraint.

### --- Block 12: TP53 threshold tuner (hold recall floor; maximize precision) ---


In [ ]:
# --- Block 12: TP53 threshold tuner (hold recall floor; maximize precision) ---
import os, json, numpy as np, pandas as pd
from sklearn.metrics import precision_recall_curve, confusion_matrix, average_precision_score, roc_auc_score, brier_score_loss

ART_DIR = "/content/drive/MyDrive/Cancer_gene_database/artifacts/final_receipts_20250816_232227"
RAW_PATH = os.path.join(ART_DIR, "preds_master.csv")
DEP_PATH = os.path.join(ART_DIR, "preds_master_DEPLOY.csv")
TH_PATH  = os.path.join(ART_DIR, "thresholds.json")

assert os.path.exists(RAW_PATH) and os.path.exists(DEP_PATH) and os.path.exists(TH_PATH)

# 1) load + LOCO align
raw = pd.read_csv(RAW_PATH); raw.columns = [c.lower() for c in raw.columns]
dep = pd.read_csv(DEP_PATH); dep.columns = [c.lower() for c in dep.columns]
if "fold" in raw.columns:
    loco = raw["fold"].astype(str).str.contains(r"\bloco[_-]?gene\b", case=False, na=False)
    if not loco.any():
        loco = raw["fold"].astype(str).str.contains("loco", case=False, na=False)
    raw = raw[loco].reset_index(drop=True)
    dep  = dep.loc[raw.index].reset_index(drop=True)

# columns
def pick_col(df, cands):
    for c in cands:
        if c in df.columns: return c
    for c in df.columns:
        if any(k in c.lower() for k in cands): return c
    return None

GENE_CANDS = ["gene","gene_symbol","target_gene","locus_gene","symbol"]
LAB_CANDS  = ["label","y","y_true","target","truth","is_driver","oncogenic"]
gene_col = pick_col(raw, GENE_CANDS);  assert gene_col
lab_col  = pick_col(raw, LAB_CANDS);   assert lab_col

def coerce_label(s):
    if s.dtype == bool: return s.astype(int)
    if s.dtype.kind in "if":
        u = pd.unique(s.dropna())
        if set(u).issubset({0,1}): return s.astype(int)
        return (s >= 0.5).astype(int)
    m = {"true":1,"yes":1,"driver":1,"oncogenic":1,"positive":1,
         "false":0,"no":0,"benign":0,"negative":0,"passenger":0}
    return s.astype(str).str.strip().str.lower().map(m).fillna(0).astype(int)

df = pd.DataFrame({
    "gene": raw[gene_col].astype(str).str.upper(),
    "y": coerce_label(raw[lab_col]),
    "p": pd.to_numeric(dep["proba_deploy"], errors="coerce").clip(0,1)
})
tp = df[df.gene=="TP53"].copy()
y, p = tp.y.values, tp.p.values

# 2) sweep thresholds by recall floors → pick best precision
def tune_by_recall_floor(y, p, floor=0.90):
    prec, rec, thr = precision_recall_curve(y, p)
    # keep indices where recall >= floor; pick argmax precision (tie → higher recall)
    idx = np.where(rec >= floor)[0]
    if idx.size == 0:
        return None  # cannot achieve that recall
    best = idx[np.argmax(prec[idx])]
    t = thr[best] if best < len(thr) else 0.5
    return float(t), float(prec[best]), float(rec[best])

floors = [0.95, 0.90, 0.85, 0.80, 0.75]
rows = []
for f in floors:
    res = tune_by_recall_floor(y, p, f)
    if res is None:
        rows.append({"recall_floor": f, "threshold": None, "precision": None, "recall": None,
                     "tp": None, "fp": None, "fn": None, "tn": None, "f1": None})
        continue
    t, pr, rc = res
    yhat = (p >= t).astype(int)
    tn, fp, fn, tp_ = confusion_matrix(y, yhat, labels=[0,1]).ravel()
    prec = tp_/(tp_+fp) if (tp_+fp)>0 else 0.0
    rec_ = tp_/(tp_+fn) if (tp_+fn)>0 else 0.0
    f1   = 2*prec*rec_/(prec+rec_) if (prec+rec_)>0 else 0.0
    rows.append({
        "recall_floor": f,
        "threshold": round(t,6),
        "precision": round(prec,4),
        "recall": round(rec_,4),
        "f1": round(f1,4),
        "tp": int(tp_), "fp": int(fp), "fn": int(fn), "tn": int(tn),
    })
tp53_grid = pd.DataFrame(rows)
print("TP53 recall-floor sweep:")
print(tp53_grid.to_string(index=False))

# 3) choose f=0.90 by default (edit here if you prefer different floor)
choice = tp53_grid.dropna().sort_values(by=["recall_floor"], ascending=False)
pick = choice[choice["recall"] >= 0.90].iloc[0] if (choice["recall"] >= 0.90).any() else choice.iloc[-1]
new_thr = float(pick["threshold"]); new_pol = f"Recall>={pick['recall_floor']:.2f}"

# 4) update thresholds.json and write a small delta file
with open(TH_PATH, "r") as f:
    TH = json.load(f)
old = TH.get("TP53", {})
TH["TP53"] = {"recommended": new_thr, "policy": new_pol, "alternates": old.get("alternates", {})}
with open(TH_PATH, "w") as f:
    json.dump(TH, f, indent=2)
print(f"[updated] TP53 threshold → {new_thr}  policy: {new_pol}")

# 5) (optional) show new confusion at chosen threshold
yhat = (p >= new_thr).astype(int)
tn, fp, fn, tp_ = confusion_matrix(y, yhat, labels=[0,1]).ravel()
prec = tp_/(tp_+fp) if (tp_+fp)>0 else 0.0
rec_ = tp_/(tp_+fn) if (tp_+fn)>0 else 0.0
f1   = 2*prec*rec_/(prec+rec_) if (prec+rec_)>0 else 0.0
print("TP53 @ chosen threshold:", {"tp":int(tp_), "fp":int(fp), "fn":int(fn), "tn":int(tn),
                                   "precision":round(prec,4), "recall":round(rec_,4), "f1":round(f1,4)})


Sweep says TP53 can’t trade recall for precision at that low cutoff — all TPs are already above **0.207547**, so any recall floor ≥0.75 lands on the same threshold. If we want fewer false alarms, we **gotta raise the bar**. Best play here: **dual-threshold triage** for TP53 — a high-precision “rule-in” band plus the existing low bar for “catch-all to review.”

### --- Block 13: TP53 dual-threshold triage (rule-in + review band) ---


In [ ]:
# --- Block 13: TP53 dual-threshold triage (rule-in + review band) ---
# Goal:
#  - Keep the low threshold (recall≈1) to funnel "review" cases
#  - Add a high threshold hitting a target precision (default 0.70) for confident positives
#  - Write 3-class calls and concise receipts

import os, json, numpy as np, pandas as pd
from sklearn.metrics import precision_recall_curve, confusion_matrix, average_precision_score, roc_auc_score, brier_score_loss

ART_DIR = "/content/drive/MyDrive/Cancer_gene_database/artifacts/final_receipts_20250816_232227"
RAW_PATH = os.path.join(ART_DIR, "preds_master.csv")
DEP_PATH = os.path.join(ART_DIR, "preds_master_DEPLOY.csv")
TH_PATH  = os.path.join(ART_DIR, "thresholds.json")

OUT_3CALLS = os.path.join(ART_DIR, "preds_master_DEPLOY_called3.csv")
OUT_3SUM   = os.path.join(ART_DIR, "eval_threeway_summary.csv")

TARGET_PREC = 0.70   # <-- tweak if you want stricter/looser rule-in

# --- load + LOCO align ---
raw = pd.read_csv(RAW_PATH); raw.columns = [c.lower() for c in raw.columns]
dep = pd.read_csv(DEP_PATH); dep.columns = [c.lower() for c in dep.columns]

if "fold" in raw.columns:
    loco = raw["fold"].astype(str).str.contains(r"\bloco[_-]?gene\b", case=False, na=False)
    if not loco.any():
        loco = raw["fold"].astype(str).str.contains("loco", case=False, na=False)
    raw = raw[loco].reset_index(drop=True)
    dep  = dep.loc[raw.index].reset_index(drop=True)

def pick_col(df, cands):
    for c in cands:
        if c in df.columns: return c
    for c in df.columns:
        if any(k in c.lower() for k in cands): return c
    return None

GENE_CANDS = ["gene","gene_symbol","target_gene","locus_gene","symbol"]
LAB_CANDS  = ["label","y","y_true","target","truth","is_driver","oncogenic"]
gene_col = pick_col(raw, GENE_CANDS);  assert gene_col, "no gene column"
lab_col  = pick_col(raw, LAB_CANDS);   assert lab_col,  "no label column"
assert "proba_deploy" in dep.columns, "DEPLOY missing proba_deploy"

def coerce_label(s):
    if s.dtype == bool: return s.astype(int)
    if s.dtype.kind in "if":
        u = pd.unique(s.dropna())
        if set(u).issubset({0,1}): return s.astype(int)
        return (s >= 0.5).astype(int)
    m = {"true":1,"yes":1,"driver":1,"oncogenic":1,"positive":1,
         "false":0,"no":0,"benign":0,"negative":0,"passenger":0}
    return s.astype(str).str.strip().str.lower().map(m).fillna(0).astype(int)

df = pd.DataFrame({
    "gene": raw[gene_col].astype(str).str.upper(),
    "y": coerce_label(raw[lab_col]),
    "p": pd.to_numeric(dep["proba_deploy"], errors="coerce").clip(0,1)
})

# --- get current TP53 low threshold from thresholds.json ---
with open(TH_PATH, "r") as f:
    TH = json.load(f)
TP53_LOW = float(TH.get("TP53", {}).get("recommended", 0.207547))

# --- compute TP53 high-threshold for target precision ---
tp = df[df.gene=="TP53"].copy()
y, p = tp.y.values, tp.p.values
prec, rec, thr = precision_recall_curve(y, p)

# candidates with precision >= target; prefer higher recall, then higher threshold
idx = np.where(prec[:-1] >= TARGET_PREC)[0]  # exclude last point (no threshold)
if idx.size == 0:
    # unreachable target: fall back to best F1 on PR curve
    f1 = np.where((prec+rec)>0, 2*prec*rec/(prec+rec), 0)
    best = int(np.nanargmax(f1[:-1]))
    TP53_HIGH = float(thr[best])
    note = f"(fallback F1-opt, target precision {TARGET_PREC} unreachable)"
else:
    # pick the index with max recall; tie-break by larger threshold (more conservative)
    best = idx[np.lexsort((-thr[idx], rec[idx]))][-1]
    TP53_HIGH = float(thr[best])
    note = f"(precision≥{TARGET_PREC:.2f})"

# --- update thresholds.json with a band for TP53 ---
TH["TP53_band"] = {"low": TP53_LOW, "high": TP53_HIGH, "policy": f"triage_low+high {note}"}
with open(TH_PATH, "w") as f:
    json.dump(TH, f, indent=2)
print(f"[band] TP53 low={TP53_LOW:.6f}, high={TP53_HIGH:.6f} {note}")

# --- apply 3-class calling ---
def call_threeway(g, p):
    if g == "TP53":
        if p >= TP53_HIGH: return "pos_high"
        if p >= TP53_LOW:  return "review"
        return "neg"
    # other genes: keep binary with their recommended threshold
    info = TH.get(g, {})
    t = float(info.get("recommended", 0.5)) if isinstance(info, dict) else 0.5
    return "pos_high" if p >= t else "neg"

df3 = dep.copy()
df3["gene"] = df["gene"]
df3["label"] = df["y"]
df3["proba_deploy"] = df["p"]
df3["call3"] = [call_threeway(g, pp) for g, pp in zip(df["gene"], df["p"])]
df3.to_csv(OUT_3CALLS, index=False)
print(f"[ok] wrote {OUT_3CALLS} rows={len(df3)}")

# --- receipts: per-gene counts; TP53 binary metrics for 'pos_high' only, and 'pos_high+review' as positive ---
rows=[]
for g, gdf in df3.groupby("gene"):
    cnts = gdf["call3"].value_counts().to_dict()
    row = {"gene": g, "n": int(len(gdf)),
           "pos_high": int(cnts.get("pos_high",0)),
           "review": int(cnts.get("review",0)),
           "neg": int(cnts.get("neg",0))}
    if g == "TP53":
        # binary eval treating pos_high as positive; review as negative
        y = gdf["label"].values
        yhat_hi = (gdf["call3"].values == "pos_high").astype(int)
        tn, fp, fn, tp = confusion_matrix(y, yhat_hi, labels=[0,1]).ravel()
        prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
        rec  = tp/(tp+fn) if (tp+fn)>0 else 0.0
        f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
        row.update({
            "TP53_high_precision": round(prec,4),
            "TP53_high_recall": round(rec,4),
            "TP53_high_f1": round(f1,4),
            "TP53_tp": int(tp), "TP53_fp": int(fp), "TP53_fn": int(fn), "TP53_tn": int(tn)
        })
        # optional: treat pos_high+review as positive (ceiling on recall if review is worked up)
        yhat_any = ((gdf["call3"].values != "neg")).astype(int)
        tn2, fp2, fn2, tp2 = confusion_matrix(y, yhat_any, labels=[0,1]).ravel()
        rec_any  = tp2/(tp2+fn2) if (tp2+fn2)>0 else 0.0
        prec_any = tp2/(tp2+fp2) if (tp2+fp2)>0 else 0.0
        row.update({
            "TP53_any_recall": round(rec_any,4),
            "TP53_any_precision": round(prec_any,4),
            "TP53_any_tp": int(tp2), "TP53_any_fp": int(fp2)
        })
    rows.append(row)

pd.DataFrame(rows).sort_values("gene").to_csv(OUT_3SUM, index=False)
print(f"[ok] wrote {OUT_3SUM}")


TP53 got a clean two-band: **low=0.207547** (catch-all) and **high=0.50** (≈70% precision rule-in). Let’s lock receipts and make this deploy-ready in one tidy move.

### Block 14 — package final “receipts” + quick sanity print

This will:

* Load our final evals + thresholds (incl. TP53 band).

* Print the three-way summary (so we can eyeball counts).

* Write a single `deploy_manifest.json` with: per-gene strategy, calibration, thresholds, and final metrics.

* Drop a concise `README_DEPLOY.md`

In [ ]:
# --- Block 14: package deploy receipts (manifest + README) and print triage summary ---
import os, json, pandas as pd
from datetime import datetime

ROOT = "/content/drive/MyDrive/Cancer_gene_database"
ART  = f"{ROOT}/artifacts/final_receipts_20250816_232227"
CFG  = f"{ROOT}/config"

# inputs
PATHS = {
    "deploy_cfg": f"{CFG}/deploy_config.json",
    "thresholds": f"{ART}/thresholds.json",
    "eval_final": f"{ART}/eval_calibration_summary_LOCO_FINAL.csv",
    "eval_threeway": f"{ART}/eval_threeway_summary.csv",
    "preds_called": f"{ART}/preds_master_DEPLOY_called.csv",
    "preds_called3": f"{ART}/preds_master_DEPLOY_called3.csv",
}

for k,p in PATHS.items():
    assert os.path.exists(p), f"missing: {k} -> {p}"

# load stuff
deploy_cfg   = json.load(open(PATHS["deploy_cfg"]))
thresholds   = json.load(open(PATHS["thresholds"]))
eval_final   = pd.read_csv(PATHS["eval_final"])
threeway     = pd.read_csv(PATHS["eval_threeway"])
called       = pd.read_csv(PATHS["preds_called"])
called3      = pd.read_csv(PATHS["preds_called3"])

# print the three-way summary for quick eyeballs
print("=== Three-way triage summary (per gene) ===")
print(threeway.to_string(index=False))

# assemble per-gene manifest entries
def safe(v):
    try:
        return None if pd.isna(v) else float(v)
    except Exception:
        return v

manifest = {
    "version": "2025-08-18",
    "generated_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "notes": [
        "EGFR/KRAS: pair-only + Platt; thresholds via YoudenJ.",
        "PIK3CA: component-cal (WT isotonic, Δ Platt) + max; threshold via Recall>=0.80.",
        "TP53: final-stage isotonic on max(WT,Δ); dual-threshold triage band (low=catch-all, high=rule-in)."
    ],
    "genes": {}
}

# pull final metrics
metrics = {}
for _, r in eval_final.iterrows():
    g = str(r["gene"]).upper()
    metrics[g] = {
        "n": int(r["n"]),
        "pos": int(r["pos"]),
        "neg": int(r["neg"]),
        "AUROC_final": safe(r["AUROC_final"]),
        "PR_final": safe(r["PR_final"]),
        "Brier_final": safe(r["Brier_final"]),
        "ECE_final": safe(r["ECE_final"]),
    }

# binary call metrics at chosen thresholds
bin_metrics = {}
for _, r in pd.read_csv(f"{ART}/eval_operating_points_summary.csv").iterrows():
    g = str(r["gene"]).upper()
    bin_metrics[g] = {
        "threshold": float(r["threshold"]),
        "policy": r["policy"],
        "tp": int(r["tp"]), "fp": int(r["fp"]), "fn": int(r["fn"]), "tn": int(r["tn"]),
        "precision": float(r["precision"]), "recall": float(r["recall"]),
        "f1": float(r["f1"]), "bacc": float(r["bacc"]),
        "auroc": None if str(r["auroc"])=="nan" else float(r["auroc"]),
        "pr_auc": float(r["pr_auc"]),
        "brier": float(r["brier"]),
    }

# three-way counts + TP53 high-band quality
three_counts = {g: grp["call3"].value_counts().to_dict()
                for g, grp in called3.groupby(called3["gene"].str.upper())}

def tp53_high_band_quality(df3):
    import numpy as np
    tp = df3[df3["gene"].str.upper()=="TP53"].copy()
    if tp.empty: return {}
    y = tp["label"].values
    hi = (tp["call3"].values=="pos_high").astype(int)
    # metrics on high band only:
    from sklearn.metrics import confusion_matrix
    tn, fp, fn, tpv = confusion_matrix(y, hi, labels=[0,1]).ravel()
    prec = tpv/(tpv+fp) if (tpv+fp)>0 else 0.0
    rec  = tpv/(tpv+fn) if (tpv+fn)>0 else 0.0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
    return {"high_precision": round(prec,4), "high_recall": round(rec,4), "high_f1": round(f1,4),
            "tp": int(tpv), "fp": int(fp), "fn": int(fn), "tn": int(tn)}

tp53_hi = tp53_high_band_quality(called3)

# build manifest.genes
for g in sorted(metrics.keys()):
    entry = {
        "strategy": deploy_cfg.get(g, deploy_cfg.get("_fallback", {})).get("strategy","unknown"),
        "calibration": deploy_cfg.get(g, {}).get("calibration", "component_cal_or_final_iso"),
        "final_metrics": metrics[g],
        "binary_operating_point": bin_metrics.get(g, {}),
        "three_way_counts": three_counts.get(g, {}),
        "thresholds": thresholds.get(g, thresholds.get(f"{g}_band", {}))
    }
    if g=="TP53" and "TP53_band" in thresholds:
        entry["triage_band"] = thresholds["TP53_band"]
        entry["tp53_high_band_quality"] = tp53_hi
    manifest["genes"][g] = entry

# write artifacts
MANIFEST_PATH = f"{ART}/deploy_manifest.json"
with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2)
print(f"\n[ok] wrote {MANIFEST_PATH}")

# README
readme = f"""# Deploy Receipts — {manifest['version']}

**Strategies**
- EGFR / KRAS → Δ-only (pair) + Platt; operating point: YoudenJ
- PIK3CA → WT isotonic + Δ Platt, then `max`; operating point: Recall≥0.80
- TP53 → final-stage isotonic on `max(WT,Δ)`; **dual-threshold**: low=review catch-all, high=rule-in (≈70% precision)

**Key final metrics (LOCO-GENE)**
{eval_final[['gene','AUROC_final','PR_final','Brier_final','ECE_final']].to_string(index=False)}

**Binary operating points @ recommended thresholds**
{pd.read_csv(f"{ART}/eval_operating_points_summary.csv")[['gene','threshold','policy','precision','recall','f1','bacc']].to_string(index=False)}

**TP53 triage band**
- low = {thresholds.get('TP53_band',{}).get('low', 'NA')}
- high = {thresholds.get('TP53_band',{}).get('high', 'NA')}
- high-band precision/recall/f1 = {tp53_hi.get('high_precision','NA')}/{tp53_hi.get('high_recall','NA')}/{tp53_hi.get('high_f1','NA')}
- counts: {three_counts.get('TP53',{})}

_Manifest JSON_: `{MANIFEST_PATH}`
"""
RD_PATH = f"{ART}/README_DEPLOY.md"
with open(RD_PATH, "w") as f:
    f.write(readme)
print(f"[ok] wrote {RD_PATH}")


### NOTE: **NaNs in triage summary**: those columns are **TP53-only** metrics (high-band precision/recall/etc.). For EGFR/KRAS/PIK3CA they don’t apply → NaN is expected. I gave you a cleaner view that hides them for non-TP53.

next bite is the **deploy-time scorer** so any CSV with `gene`, `wt_score`, `pair_score` in → calibrated **proba + call** (incl. TP53 triage). drop this in as a single cell.

### --- Block 15: deploy-time inference utility (load calibrators + thresholds, score CSVs) ---


In [ ]:
# --- Block 15: deploy-time inference utility (load calibrators + thresholds, score CSVs) ---
# Writes a reusable module and demo-calls it. No drive remounts, just files.
import os, re, json, joblib, pandas as pd, numpy as np
from datetime import datetime

ROOT_CFG = "/content/drive/MyDrive/Cancer_gene_database/config"
CAL_DIR  = os.path.join(ROOT_CFG, "calibration")
DEPLOY_CFG_PATH = os.path.join(ROOT_CFG, "deploy_config.json")
THRESH_JSON     = "/content/drive/MyDrive/Cancer_gene_database/artifacts/final_receipts_20250816_232227/thresholds.json"

SRC_DIR = "/content/drive/MyDrive/Cancer_gene_database/src"
os.makedirs(SRC_DIR, exist_ok=True)
MOD_PATH = os.path.join(SRC_DIR, "inference_utils.py")

code = r'''
import os, re, json, joblib, pandas as pd, numpy as np

# column sniffers
GENE_CANDS  = ["gene","gene_symbol","target_gene","locus_gene","symbol"]
WT_CANDS    = ["wt_score","wt_prob","prob_wt","score_wt","wt","base_prob","base_score","wtonly","wt_only"]
PAIR_CANDS  = ["pair_score","pair_prob","prob_pair","score_pair","delta_score","delta_prob",
               "dcnn_prob","cnn_prob","ranknet_prob","twor_prob","pair","delta","dcnn","cnn","twor"]

def _pick_col(df, cands):
    for c in cands:
        if c in df.columns: return c
    for c in df.columns:
        if any(k in c.lower() for k in cands): return c
    return None

class InferenceEngine:
    """
    Gene-aware scorer:
      - EGFR/KRAS: Δ-only (Platt) → threshold (YoudenJ in thresholds.json)
      - PIK3CA: WT isotonic + Δ Platt → max → threshold Recall>=0.80
      - TP53:  final-stage isotonic on max(WT_raw, Δ_raw); dual-threshold band (low/high)
    """
    def __init__(self, deploy_cfg_path, cal_dir, thresholds_json):
        with open(deploy_cfg_path, "r") as f:
            self.deploy = json.load(f)
        self.cal = {}
        if os.path.isdir(cal_dir):
            for p in os.listdir(cal_dir):
                m = re.match(r"cal_([A-Za-z0-9]+)_(wt|pair)\.pkl$", p)
                if m:
                    g, k = m.group(1).upper(), m.group(2)
                    self.cal[(g,k)] = joblib.load(os.path.join(cal_dir, p))
        # TP53 final-stage iso
        self.tp53_final = None
        tp53_final_path = os.path.join(cal_dir, "cal_TP53_final_iso.pkl")
        if os.path.exists(tp53_final_path):
            self.tp53_final = joblib.load(tp53_final_path)

        with open(thresholds_json, "r") as f:
            self.th = json.load(f)

    def _apply_component_cal(self, gene, kind, score):
        if score is None or (isinstance(score, float) and np.isnan(score)):
            return None
        model = self.cal.get((gene.upper(), kind))
        if model is None:
            return float(score)
        try:
            if hasattr(model, "transform"):      # IsotonicRegression
                return float(model.transform([float(score)])[0])
            if hasattr(model, "predict_proba"):  # LogisticRegression (Platt)
                return float(model.predict_proba([[float(score)]])[0,1])
        except Exception:
            return float(score)
        return float(score)

    def _tp53_final(self, score_pre):
        if score_pre is None or (isinstance(score_pre, float) and np.isnan(score_pre)):
            return 0.5
        if self.tp53_final is None:
            return float(score_pre)
        try:
            return float(self.tp53_final.transform([float(score_pre)])[0])
        except Exception:
            return float(score_pre)

    def predict_proba(self, gene, wt_raw=None, pair_raw=None):
        g = str(gene).upper()
        cfg = self.deploy.get(g, self.deploy.get("_fallback", {"strategy":"blend","blend_weights":{"wt":0.5,"pair":0.5}}))
        strat = cfg["strategy"]

        # Special-case TP53: final-stage iso on pre-score
        if g == "TP53":
            pre = max(wt_raw if wt_raw is not None else -1, pair_raw if pair_raw is not None else -1)
            return self._tp53_final(pre)

        # Δ-only genes
        if g in {"EGFR","KRAS"} or strat == "pair":
            pair_c = self._apply_component_cal(g, "pair", pair_raw)
            if pair_c is not None:
                return float(pair_c)
            # fallback
            wt_c = self._apply_component_cal(g, "wt", wt_raw)
            return float(wt_c) if wt_c is not None else 0.5

        # context-independent / max
        if g == "PIK3CA" or strat == "max":
            wt_c   = self._apply_component_cal(g, "wt", wt_raw)
            pair_c = self._apply_component_cal(g, "pair", pair_raw)
            if wt_c is None and pair_c is None: return 0.5
            if wt_c is None: return float(pair_c)
            if pair_c is None: return float(wt_c)
            return float(max(wt_c, pair_c))

        # generic blend fallback
        if strat == "blend":
            w = cfg.get("blend_weights", {"wt":0.5,"pair":0.5})
            wt_c   = self._apply_component_cal(g, "wt", wt_raw)   or 0.0
            pair_c = self._apply_component_cal(g, "pair", pair_raw) or 0.0
            return float(w["wt"]*wt_c + w["pair"]*pair_c)

        # last resort
        pair_c = self._apply_component_cal(g, "pair", pair_raw)
        if pair_c is not None: return float(pair_c)
        wt_c = self._apply_component_cal(g, "wt", wt_raw)
        return float(wt_c) if wt_c is not None else 0.5

    def call_for_gene(self, gene, proba):
        g = str(gene).upper()
        # TP53 triage band
        if g == "TP53" and "TP53_band" in self.th:
            low  = float(self.th["TP53_band"]["low"])
            high = float(self.th["TP53_band"]["high"])
            if proba >= high: return "pos_high", high
            if proba >= low:  return "review", low
            return "neg", low
        # binary thresholds
        info = self.th.get(g, {})
        t = float(info.get("recommended", 0.5)) if isinstance(info, dict) else 0.5
        return ("pos_high" if proba >= t else "neg"), t

    def score_dataframe(self, df):
        # sniff columns
        cols = [c for c in df.columns]
        gene_col = _pick_col(df, GENE_CANDS)
        if gene_col is None:
            raise ValueError(f"Need a gene column. Saw: {cols}")
        wt_col   = _pick_col(df, WT_CANDS)
        pair_col = _pick_col(df, PAIR_CANDS)
        if wt_col is None and pair_col is None:
            raise ValueError(f"Need at least one of WT or PAIR scores. Saw: {cols}")

        out = df.copy()
        proba = []
        calls = []
        th_used = []
        for g, w, p in zip(out[gene_col].astype(str),
                           out[wt_col] if wt_col else [None]*len(out),
                           out[pair_col] if pair_col else [None]*len(out)):
            wv = float(w) if w is not None and str(w)!="nan" else None
            pv = float(p) if p is not None and str(p)!="nan" else None
            pr = self.predict_proba(g, wv, pv)
            call, thr = self.call_for_gene(g, pr)
            proba.append(pr); calls.append(call); th_used.append(thr)
        out["proba"] = proba
        out["call"] = calls
        out["threshold_used"] = th_used
        return out
'''

with open(MOD_PATH, "w") as f:
    f.write(code)

print(f"[ok] wrote module: {MOD_PATH}")

# ---- Demo: score your existing master preds (LOCO rows) and write a “deploy_scored.csv” ----
from inference_utils import InferenceEngine

engine = InferenceEngine(
    deploy_cfg_path=DEPLOY_CFG_PATH,
    cal_dir=CAL_DIR,
    thresholds_json=THRESH_JSON
)

IN_PATH  = "/content/drive/MyDrive/Cancer_gene_database/artifacts/final_receipts_20250816_232227/preds_master.csv"
OUT_PATH = "/content/drive/MyDrive/Cancer_gene_database/artifacts/final_receipts_20250816_232227/deploy_scored.csv"

df_in = pd.read_csv(IN_PATH)
# keep LOCO-GENE slice if present (matches everything we've evaluated)
if "fold" in df_in.columns:
    mask = df_in["fold"].astype(str).str.contains(r"\bloco[_-]?gene\b", case=False, na=False)
    if not mask.any():
        mask = df_in["fold"].astype(str).str.contains("loco", case=False, na=False)
    df_in = df_in[mask].reset_index(drop=True)

scored = engine.score_dataframe(df_in)
scored.to_csv(OUT_PATH, index=False)
print(f"[deploy] wrote {OUT_PATH}  rows={len(scored)}")

# show a tiny head so you can eyeball it
print(scored.head(6)[["gene","proba","call","threshold_used"]].to_string(index=False))


##Block 15.fix — add src/ to path, import, smoke-test, and pretty triage table

In [ ]:
# --- Block 15.fix: path + import + smoke test + pretty triage view ---
import os, sys, importlib.util, pandas as pd

SRC_DIR = "/content/drive/MyDrive/Cancer_gene_database/src"
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

# robust import (works even if Python hasn't indexed the new file yet)
try:
    from inference_utils import InferenceEngine
except ModuleNotFoundError:
    spec = importlib.util.spec_from_file_location("inference_utils", os.path.join(SRC_DIR, "inference_utils.py"))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    InferenceEngine = mod.InferenceEngine

# --- quick smoke test: score LOCO-GENE slice of your master preds and print head ---
ROOT_CFG = "/content/drive/MyDrive/Cancer_gene_database/config"
CAL_DIR  = os.path.join(ROOT_CFG, "calibration")
DEPLOY_CFG_PATH = os.path.join(ROOT_CFG, "deploy_config.json")
THRESH_JSON = "/content/drive/MyDrive/Cancer_gene_database/artifacts/final_receipts_20250816_232227/thresholds.json"

engine = InferenceEngine(DEPLOY_CFG_PATH, CAL_DIR, THRESH_JSON)

IN_PATH  = "/content/drive/MyDrive/Cancer_gene_database/artifacts/final_receipts_20250816_232227/preds_master.csv"
OUT_PATH = "/content/drive/MyDrive/Cancer_gene_database/artifacts/final_receipts_20250816_232227/deploy_scored.csv"

df_in = pd.read_csv(IN_PATH)
if "fold" in df_in.columns:
    mask = df_in["fold"].astype(str).str.contains(r"\bloco[_-]?gene\b", case=False, na=False)
    if not mask.any():
        mask = df_in["fold"].astype(str).str.contains("loco", case=False, na=False)
    df_in = df_in[mask].reset_index(drop=True)

scored = engine.score_dataframe(df_in)
scored.to_csv(OUT_PATH, index=False)
print(f"[deploy] wrote {OUT_PATH}  rows={len(scored)}")
print(scored.head(6)[["gene","proba","call","threshold_used"]].to_string(index=False))

# --- pretty three-way triage view (hide TP53-only metrics for other genes) ---
TW_PATH = "/content/drive/MyDrive/Cancer_gene_database/artifacts/final_receipts_20250816_232227/eval_threeway_summary.csv"
tw = pd.read_csv(TW_PATH)

# 1) counts for all genes
counts = tw[["gene","n","pos_high","review","neg"]].sort_values("gene").reset_index(drop=True)
print("\n=== Three-way counts (all genes) ===")
print(counts.to_string(index=False))

# 2) TP53 high-band quality only (no NaNs)
tp53 = tw[tw["gene"].str.upper()=="TP53"]
if not tp53.empty:
    keep_cols = [c for c in tp53.columns if c.startswith("TP53_")]
    tp53_view = tp53[["gene"] + keep_cols].reset_index(drop=True)
    print("\n=== TP53 high-band quality ===")
    print(tp53_view.to_string(index=False))


everything’s wired and spitting the right receipts. Next smart bite is an audit pack so future-us can eyeball whether these probs are trustworthy at a glance.

### Block 16 — audit pack (pretty CSVs + reliability & PR plots per gene)

This will:

* make **pretty CSVs** (counts-only for all genes, TP53 triage metrics separate — no NaNs)

* plot **calibration curves** (pre vs final) and **PR curves** per gene

* drop PNGs in a `plots/` folder next to receipts

In [ ]:
# --- Block 16: audit pack (pretty CSVs + reliability & PR plots) ---
import os, json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

ROOT = "/content/drive/MyDrive/Cancer_gene_database"
ART  = f"{ROOT}/artifacts/final_receipts_20250816_232227"
CFG  = f"{ROOT}/config"
os.makedirs(f"{ART}/plots", exist_ok=True)

# ---------- 1) Pretty CSVs (no NaNs for non-TP53) ----------
tw_path = f"{ART}/eval_threeway_summary.csv"
tw = pd.read_csv(tw_path)

counts = tw[["gene","n","pos_high","review","neg"]].sort_values("gene").reset_index(drop=True)
counts_path = f"{ART}/eval_threeway_counts.csv"
counts.to_csv(counts_path, index=False)

tp53 = tw[tw["gene"].str.upper()=="TP53"].copy()
tp53_cols = [c for c in tp53.columns if c.startswith("TP53_")]
tp53_pretty = tp53[["gene"] + tp53_cols].reset_index(drop=True)
tp53_pretty_path = f"{ART}/eval_tp53_triage_metrics.csv"
tp53_pretty.to_csv(tp53_pretty_path, index=False)

print("[ok] wrote", counts_path)
print("[ok] wrote", tp53_pretty_path)

# ---------- 2) Reliability (pre vs final) + PR curves ----------
RAW_PATH   = f"{ART}/preds_master.csv"
FINAL_PATH = f"{ART}/preds_master_finalcal_cv.csv"  # has score_final_v3
DEPLOY_CFG = f"{CFG}/deploy_config.json"

assert os.path.exists(RAW_PATH) and os.path.exists(FINAL_PATH) and os.path.exists(DEPLOY_CFG)

with open(DEPLOY_CFG, "r") as f:
    DEPLOY = json.load(f)

raw   = pd.read_csv(RAW_PATH)
final = pd.read_csv(FINAL_PATH)
raw.columns = [c.lower() for c in raw.columns]
final.columns = [c.lower() for c in final.columns]

# LOCO-GENE alignment
if "fold" in raw.columns:
    loco = raw["fold"].astype(str).str.contains(r"\bloco[_-]?gene\b", case=False, na=False)
    if not loco.any():
        loco = raw["fold"].astype(str).str.contains("loco", case=False, na=False)
    raw   = raw[loco].reset_index(drop=True)
    final = final.loc[raw.index].reset_index(drop=True)

# column sniffers
LABEL_CANDS = ["label","y","y_true","target","truth","is_driver","oncogenic"]
GENE_CANDS  = ["gene","gene_symbol","target_gene","locus_gene","symbol"]
WT_CANDS    = ["wt_score","wt_prob","prob_wt","score_wt","wt","base_prob","base_score","wtonly","wt_only"]
PAIR_CANDS  = ["pair_score","pair_prob","prob_pair","score_pair","delta_score","delta_prob","dcnn_prob","cnn_prob","ranknet_prob","twor_prob","pair","delta","dcnn","cnn","twor"]

def pick_col(df, cands):
    for c in cands:
        if c in df.columns: return c
    for c in df.columns:
        if any(k in c.lower() for k in cands): return c
    return None

def coerce_label(s):
    if s.dtype == bool: return s.astype(int)
    if s.dtype.kind in "if":
        u = pd.unique(s.dropna())
        if set(u).issubset({0,1}): return s.astype(int)
        return (s >= 0.5).astype(int)
    m = {"true":1,"yes":1,"driver":1,"oncogenic":1,"positive":1,
         "false":0,"no":0,"benign":0,"negative":0,"passenger":0}
    return s.astype(str).str.strip().str.lower().map(m).fillna(0).astype(int)

gene_col  = pick_col(raw, GENE_CANDS);  assert gene_col
label_col = pick_col(raw, LABEL_CANDS); assert label_col
wt_col    = pick_col(raw, WT_CANDS)
pair_col  = pick_col(raw, PAIR_CANDS)

# pre-score = combine per deploy strategy (before any calibration)
def combine_pre(gene, wt, pair):
    cfg = DEPLOY.get(gene, DEPLOY.get("_fallback"))
    strat = cfg["strategy"]
    if strat == "pair":
        return float(pair if pd.notna(pair) else (wt if pd.notna(wt) else 0.5))
    if strat == "max":
        a = wt if pd.notna(wt) else -1
        b = pair if pd.notna(pair) else -1
        return float(max(a, b))
    if strat == "blend":
        w = cfg.get("blend_weights", {"wt":0.5,"pair":0.5})
        a = wt if pd.notna(wt) else 0.0
        b = pair if pd.notna(pair) else 0.0
        return float(w["wt"]*a + w["pair"]*b)
    return float(0.5*((wt if pd.notna(wt) else 0.0) + (pair if pd.notna(pair) else 0.0)))

df = pd.DataFrame({
    "gene": raw[gene_col].astype(str).str.upper(),
    "y": coerce_label(raw[label_col]),
})
df["wt_raw"]   = pd.to_numeric(raw[wt_col], errors="coerce") if wt_col else np.nan
df["pair_raw"] = pd.to_numeric(raw[pair_col], errors="coerce") if pair_col else np.nan
df["pre"]      = [combine_pre(g, df.at[i,"wt_raw"], df.at[i,"pair_raw"]) for i, g in enumerate(df["gene"])]
df["final"]    = pd.to_numeric(final.get("score_final_v3", final.get("score_final", df["pre"])), errors="coerce").clip(0,1)

# plotting helpers
def reliability_xy(y, p, bins=10):
    y = np.asarray(y); p = np.asarray(p)
    edges = np.unique(np.quantile(p, np.linspace(0,1,bins+1)))
    xs, ys = [], []
    for i in range(len(edges)-1):
        lo, hi = edges[i], edges[i+1] + (1e-12 if i==len(edges)-2 else 0)
        m = (p>=lo) & (p<hi)
        if not m.any(): continue
        xs.append(p[m].mean()); ys.append(y[m].mean())
    return np.array(xs), np.array(ys)

for g, gdf in df.groupby("gene"):
    y = gdf["y"].values
    pre = gdf["pre"].values
    fin = gdf["final"].values
    bins = min(10, max(5, len(y)//10))

    # --- Reliability plot (pre vs final) ---
    x1,y1 = reliability_xy(y, pre, bins=bins)
    x2,y2 = reliability_xy(y, fin, bins=bins)

    plt.figure(figsize=(5,5))
    plt.plot([0,1],[0,1], linestyle="--")
    plt.plot(x1,y1, marker="o", label="pre")
    plt.plot(x2,y2, marker="s", label="final")
    plt.xlabel("Predicted probability")
    plt.ylabel("Empirical positive rate")
    plt.title(f"{g} — Reliability (pre vs final)")
    plt.legend()
    rel_path = f"{ART}/plots/{g}_reliability.png"
    plt.savefig(rel_path, bbox_inches="tight", dpi=150)
    plt.close()
    print("[plot] wrote", rel_path)

    # --- PR curve (final only) ---
    prec, rec, _ = precision_recall_curve(y, fin)
    plt.figure(figsize=(5,5))
    plt.plot(rec, prec)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"{g} — PR curve (final)")
    pr_path = f"{ART}/plots/{g}_pr_curve.png"
    plt.savefig(pr_path, bbox_inches="tight", dpi=150)
    plt.close()
    print("[plot] wrote", pr_path)


# Day Wrap — “Sentiment-of-Sequence” Progress & Where We’re Headed

**Thesis we’re driving toward (and testing):**  
Oncogenicity speaks in **two grammars** encoded in sequence context:
- **Context-independent (fragility grammar):** the **WT context alone** flags regions where *any* change is risky (tumor suppressor vibe).
- **Context-dependent (activation grammar):** the **$Δ$/mutation itself** carries the signal (oncogene vibe).

Today we pushed this from vibes → receipts with leak-free **LOCO-GENE** evals, calibration audits, and deployable rules.

---

## What we validated today (skeptical read)

**1) Grammar type per gene (LOCO-GENE):**
- **EGFR, KRAS → context-dependent.** Pair/$Δ$ models beat WT (e.g., EGFR AUROC_pair ≈ **0.78** vs WT ~ **0.61**; KRAS $Δ$ ≈ **0.88** vs WT ~ **0.42**).
- **PIK3CA, TP53 → context-independent.** WT beats/survives vs pair (PIK3CA WT ≈ **0.76** > $Δ$ ~ **0.72**; TP53 WT ≈ **0.72** > $Δ$ ~ **0.65**).  
*Translation:* the data itself split genes into **activation** vs **fragility** modes — exactly our hypothesis.

**2) Calibration matters (and where):**
- **EGFR:** AUROC/PR stable (as expected); **Brier 0.36 → 0.24**, **ECE 0.52 → 0.22** after Δ-Platt.  
- **KRAS (tiny-n caution):** AUROC/PR stable; **Brier 0.72 → 0.26**, **ECE 0.91 → 0.26** (calm those wild probs).  
- **PIK3CA:** component-cal (WT-isotonic + Δ-Platt) then **max**: **AUROC 0.76 → 0.82**, **Brier 0.19 → 0.14**, **ECE 0.15 → 0.02**.  
- **TP53:** component-cal **hurt** (max flips winners). Fix: **final-stage isotonic on `max(WT,Δ)`**.  
  - Cross-validated final used for eval: **AUROC 0.7066**, **PR 0.5541**, **Brier 0.1922**, **ECE 0.064** (vs pre Brier 0.218/ECE 0.194).  
  - Full-fit calibrator saved for deploy.  
*Bottom line:* **calibrate where the signal lives** ($Δ$ for EGFR/KRAS, WT/max for PIK3CA/TP53). Never calibrate components then take **`max`** blindly.

**3) Decision rules that actually ship:**
- **EGFR/KRAS:** **Δ-only + Platt**, threshold via **YoudenJ**.  
  - EGFR @ 0.4666 → **P=0.61**, **R=0.79**, **F1=0.69**.  
  - KRAS @ 0.4596 → **P=0.88**, **R=0.88**, **F1=0.88** *(tiny-n, keep humility)*.  
- **PIK3CA:** WT-iso & $Δ$-Platt → **max**, **Recall≥0.80** op point @ 0.625 → **P=R=0.83**.  
- **TP53:** single threshold with **Recall≥0.95** lands at 0.2075 → **R=1.00**, **P=0.37** (intentional “screen-then-verify”).  
  - Added **dual-threshold triage band**: **low=0.2075 (review)**, **high=0.50 (rule-in ≈70% precision)**.  
  - TP53 counts: **pos_high=54**, **review=131**, **neg=12**; high-band **P≈0.704**, **R≈0.551**, **F1≈0.618**.

---

## Why this advances the hypothesis (not just numbers)

- We **observed the dialects**: Δ-driven (EGFR/KRAS) vs WT/max-driven (PIK3CA/TP53). That’s the central **“lingua onco”** claim showing up in out-of-gene validation, not just in-sample shine.  
- We **separated opportunity vs consequence** by seeing where calibration helps: Δ calibration improves activation genes; final-stage iso on max stabilizes fragility genes.  
- We **operationalized** the science: per-gene strategies + thresholds, and a TP53 **triage** that respects clinical reality (catch everything, rule-in the strong).

---

## Artifacts produced (so future-us can rerun/ship)

- **Calibrators:** Δ-Platt (EGFR/KRAS); WT-iso & Δ-Platt (PIK3CA); **TP53 final-iso** (`cal_TP53_final_iso.pkl`).  
- **Deploy config + thresholds:** `deploy_config.json`, `thresholds.json` (incl. **TP53_band {low, high}**).  
- **Receipts:** `eval_calibration_summary_LOCO_FINAL.csv`, `operating_points.csv`, `eval_operating_points_summary.csv`, three-way: `eval_threeway_summary.csv`.  
- **Manifest + README:** `deploy_manifest.json`, `README_DEPLOY.md`.  
- **Inference util:** `src/inference_utils.py` + `deploy_scored.csv` (proba + calls).

---

## What we do next (forward-looking, data-first)

1. **Feature lift (minimal-regret):** codon position, local GC%, palindromes/repeats, exon boundary distance, strand orientation/asymmetry.  
2. **Three-class framing:** **driver / passenger / germline-benign** to disentangle mutational **opportunity** vs **consequence**.  
3. **Grammar maps:** per-gene saliency to visualize **fragility regions** (TP53) vs **activation patterns** (EGFR/KRAS).  
4. **Stability diagnostics:** negative-set size vs performance curves; lock minimal n for reliable $Δ$ models.  
5. **Signature alignment:** correlate learned features with **COSMIC SBS signatures** to pin the chemical/repair story behind the “sentiment.”

**Net-net:** today we didn’t chase a new hypothesis — we **let the data speak** and tightened the loop: grammar types emerged, calibration/aggregation got gene-aware, and deploy logic is real. Tomorrow we widen features and stress-test the dialects. No cap.
